# CS781 Legal NER — Self-contained new experiments

This notebook reconstructs the known champion + rules pipeline, checks the established validation hybrid score (~0.333307), then runs **two new model experiments** from independent copies of the champion:

1. **Experiment A:** class-weighted cross-entropy fine-tuning.
2. **Experiment B:** class-weighted focal-loss fine-tuning (gamma=2).

Both train on fold 1 and evaluate on fold 2 using exact-span macro-F1 and the existing counsel/judge/WIT rules plus the selected CASENO+STAT boundary correction. No test gold labels are used. Existing champion files and predictions are never overwritten.

Run the notebook top-to-bottom. If the baseline reproduction check fails, training stops rather than producing misleading results.


In [ ]:
# 1. Necessary imports only + runtime
import os, json, re, gc, math, copy
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
from tokenizers import Tokenizer
from transformers import GPT2Config, GPT2Model
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Inventory all available datasets/checkpoints before choosing anything
ROOTS = [Path("/kaggle/input"), Path("/kaggle/working")]
CHECKPOINT_NAMES = {
    "champion_recovery.pt", "best_ner_model.pt", "checkpoint_latest.pt",
    "targeted_pilot_8_steps.pt", "best_model.pt", "model.pt"
}
DATASET_NAMES = {
    "legal_bpe_32k.json", "fold1-text.json", "fold1-gold.json",
    "fold2-text.json", "fold2-gold.json", "fold3-text.json",
    "train.json", "test.json"
}
all_files = []
for root in ROOTS:
    if root.exists():
        all_files.extend(p for p in root.rglob("*") if p.is_file())

all_files = sorted(set(all_files))
checkpoint_files = [p for p in all_files if p.name in CHECKPOINT_NAMES or p.suffix.lower() in {".pt", ".pth"}]
dataset_files = [p for p in all_files if p.name in DATASET_NAMES or ("fold" in p.name.lower() and p.suffix.lower() == ".json")]

print("=== CHECKPOINT INVENTORY ===")
if checkpoint_files:
    for p in checkpoint_files:
        print(f"{p}  [{p.stat().st_size / 1024**2:.1f} MiB]")
else:
    print("No checkpoint files found.")

print("\n=== DATASET/TOKENIZER INVENTORY ===")
for p in dataset_files:
    print(p)

print("\n=== BACKUP DATASET DIRECTORIES ===")
for p in sorted({str(x.parent) for x in dataset_files}):
    print(p)

# Pick the recovery champion explicitly; don't accidentally choose an arbitrary checkpoint_latest.pt.
champion_candidates = [p for p in checkpoint_files if p.name == "champion_recovery.pt"]
if not champion_candidates:
    champion_candidates = [p for p in checkpoint_files if p.name == "best_ner_model.pt"]
if not champion_candidates:
    raise FileNotFoundError(
        "No champion_recovery.pt or best_ner_model.pt found under /kaggle/input or /kaggle/working. "
        "Attach the CS781 recovery/checkpoint dataset in Kaggle."
    )

def rank_champion_path(p):
    s = str(p).lower()
    return (
        0 if "cs781_recovery" in s else 1,
        0 if "working-backup" in s else 1,
        0 if "/kaggle/input/" in s else 1,
        s
    )

CHAMPION_PATH = sorted(champion_candidates, key=rank_champion_path)[0]

phase1_candidates = [
    p for p in checkpoint_files
    if p.name == "checkpoint_latest.pt" and "cs781-phase-1-checkpoints" in str(p)
]
if not phase1_candidates:
    phase1_candidates = [
        p for p in checkpoint_files
        if p.name == "checkpoint_latest.pt" and "phase-1" in str(p).lower()
    ]
if not phase1_candidates:
    phase1_candidates = [p for p in checkpoint_files if p.name == "checkpoint_latest.pt"]

if not phase1_candidates:
    raise FileNotFoundError("No Phase-1 checkpoint_latest.pt found.")

PHASE1_PATH = sorted(phase1_candidates, key=lambda p: (0 if "/kaggle/input/" in str(p) else 1, str(p)))[0]

print("\nSELECTED NER CHAMPION:", CHAMPION_PATH)
print("SELECTED PHASE-1 ENCODER:", PHASE1_PATH)
print("Selection policy: recovery champion for NER; Phase-1 checkpoint for encoder.")

In [ ]:
# 3. Inspect checkpoint metadata; only the known champion is selected for NER inference
def safe_torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

recovery = safe_torch_load(CHAMPION_PATH)
phase1_ckpt = safe_torch_load(PHASE1_PATH)

print("Champion checkpoint keys:", list(recovery.keys()) if isinstance(recovery, dict) else type(recovery))
print("Phase-1 checkpoint keys:", list(phase1_ckpt.keys()) if isinstance(phase1_ckpt, dict) else type(phase1_ckpt))
if isinstance(phase1_ckpt, dict):
    print("Phase-1 step/epoch metadata:", {
        k: phase1_ckpt.get(k) for k in ("step", "global_step", "epoch", "best_step")
        if k in phase1_ckpt
    })

# Report other checkpoints' metadata without silently replacing the champion.
print("\n=== OTHER CHECKPOINT METADATA ===")
for p in checkpoint_files:
    if p in (CHAMPION_PATH, PHASE1_PATH):
        continue
    try:
        obj = safe_torch_load(p)
        if isinstance(obj, dict):
            keys = list(obj.keys())
            meta = {k: obj.get(k) for k in ("step", "global_step", "epoch", "optimizer_steps")
                    if k in obj and isinstance(obj.get(k), (int, float, str))}
            state = obj.get("model_state_dict")
            shape_info = None
            if isinstance(state, dict):
                cls = state.get("classifier.weight")
                if cls is not None and hasattr(cls, "shape"):
                    shape_info = tuple(cls.shape)
            print(f"{p}: keys={keys[:12]}, metadata={meta}, classifier.weight={shape_info}")
        else:
            print(f"{p}: object={type(obj).__name__}")
        del obj
    except Exception as e:
        print(f"{p}: could not inspect ({type(e).__name__}: {e})")

In [ ]:
# 4. Reconstruct model and load tokenizer/data files
BIO_LABELS = list(recovery["bio_labels"])
label2id = {label: i for i, label in enumerate(BIO_LABELS)}
id2label = {i: label for i, label in enumerate(BIO_LABELS)}
ENTITY_LABELS = [
    "APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT",
    "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
]
assert len(BIO_LABELS) == 25, f"Expected 25 BIO labels, got {len(BIO_LABELS)}"

cfg = phase1_ckpt.get("model_config")
if cfg is None:
    raise KeyError("Phase-1 checkpoint has no model_config; cannot reconstruct encoder safely.")

gpt2_config = GPT2Config(
    vocab_size=cfg["vocab_size"],
    n_positions=cfg["context_length"],
    n_ctx=cfg["context_length"],
    n_embd=cfg["hidden_size"],
    n_layer=cfg["num_layers"],
    n_head=cfg["num_heads"],
    resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0
)

class LegalNERModel(nn.Module):
    def __init__(self, pretrained_lm, num_labels):
        super().__init__()
        self.encoder = pretrained_lm.transformer
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(pretrained_lm.config.hidden_size, num_labels)
    def forward(self, input_ids, attention_mask=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.classifier(self.dropout(outputs.last_hidden_state))

phase1_lm = GPT2Model(gpt2_config)
phase1_state = {
    k.removeprefix("transformer."): v
    for k, v in phase1_ckpt["model_state_dict"].items()
    if k.startswith("transformer.")
}
phase1_lm.load_state_dict(phase1_state, strict=True)
lm_wrapper = type("LMWrapper", (), {"transformer": phase1_lm, "config": gpt2_config})()
champion_model = LegalNERModel(lm_wrapper, len(BIO_LABELS)).to(device)
champion_model.load_state_dict(recovery["model_state_dict"], strict=True)
champion_model.eval()

def find_named_file(filename, preferred_tokens=()):
    matches = [p for p in all_files if p.name == filename]
    if not matches:
        raise FileNotFoundError(f"Required file {filename} not found in Kaggle input/working.")
    preferred = [p for p in matches if all(t.lower() in str(p).lower() for t in preferred_tokens)]
    candidates = preferred or matches
    return sorted(candidates, key=lambda p: (0 if "/kaggle/input/" in str(p) else 1, str(p)))[0]

tokenizer_path = find_named_file("legal_bpe_32k.json")
fold1_text_path = find_named_file("fold1-text.json")
fold1_gold_path = find_named_file("fold1-gold.json")
fold2_text_path = find_named_file("fold2-text.json")
fold2_gold_path = find_named_file("fold2-gold.json")
fold3_text_path = find_named_file("fold3-text.json")

tokenizer = Tokenizer.from_file(str(tokenizer_path))
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_texts = load_json(fold1_text_path)
train_gold = load_json(fold1_gold_path)
val_texts = load_json(fold2_text_path)
val_gold = load_json(fold2_gold_path)
test_texts = load_json(fold3_text_path)

# Normalize document IDs to strings consistently.
train_texts = {str(k): v for k, v in train_texts.items()}
train_gold = {str(k): v for k, v in train_gold.items()}
val_texts = {str(k): v for k, v in val_texts.items()}
val_gold = {str(k): v for k, v in val_gold.items()}
test_texts = {str(k): v for k, v in test_texts.items()}

print("Champion loaded:", CHAMPION_PATH)
print("Tokenizer:", tokenizer_path, "| vocab size:", tokenizer.get_vocab_size())
print("Documents: train", len(train_texts), "| validation", len(val_texts), "| test", len(test_texts))
print("Train/val IDs disjoint:", set(train_texts).isdisjoint(val_texts))
print("Validation text/gold IDs match:", set(val_texts) == set(val_gold))
print("Train text/gold IDs match:", set(train_texts) == set(train_gold))
assert set(train_texts) == set(train_gold)
assert set(val_texts) == set(val_gold)

In [ ]:
# 5. Find and parse champion thresholds
threshold_candidates = [
    p for p in all_files
    if p.suffix.lower() == ".json" and "threshold" in p.name.lower()
]
print("Threshold files found:")
for p in threshold_candidates:
    print(" ", p)

def extract_thresholds(obj):
    if not isinstance(obj, dict):
        return None
    for key in ("champ_thr", "thresholds", "label_thresholds", "threshold_by_label"):
        if isinstance(obj.get(key), dict) and obj[key] and all(isinstance(v, (int, float)) for v in obj[key].values()):
            return {str(k): float(v) for k, v in obj[key].items()}
    if obj and all(isinstance(v, (int, float)) for v in obj.values()):
        return {str(k): float(v) for k, v in obj.items()}
    return None

champ_thr = None
threshold_path_used = None
for p in sorted(threshold_candidates, key=lambda p: (
    0 if "champion_thresholds_recovery" in p.name.lower() else 1,
    0 if "cs781_recovery" in str(p).lower() else 1,
    str(p)
)):
    try:
        obj = load_json(p)
        found = extract_thresholds(obj)
        if found and all(lab in found for lab in ENTITY_LABELS):
            champ_thr, threshold_path_used = found, p
            break
    except Exception:
        pass

if champ_thr is None:
    # These are the documented recovery thresholds from the existing notebook.
    champ_thr = {
        "APP": 0.15, "RESP": 0.45, "A.COUNSEL": 0.45, "R.COUNSEL": 0.45,
        "JUDGE": 0.25, "WIT": 0.45, "AUTH": 0.35, "COURT": 0.45,
        "STAT": 0.25, "PREC": 0.35, "DATE": 0.55, "CASENO": 0.35
    }
    print("WARNING: no usable threshold JSON found; using documented recovery thresholds.")
else:
    print("Loaded thresholds from:", threshold_path_used)
print("Thresholds:", champ_thr)

In [ ]:
# 6. Core inference, BIO decoding, and exact-span evaluation helpers
from bisect import bisect_left, bisect_right

NUM_LABELS = len(BIO_LABELS)
legal_start = np.array([tag == "O" or tag.startswith("B-") for tag in BIO_LABELS], dtype=bool)
legal_transition = np.ones((NUM_LABELS, NUM_LABELS), dtype=bool)
for prev_id, prev_tag in enumerate(BIO_LABELS):
    for curr_id, curr_tag in enumerate(BIO_LABELS):
        if curr_tag.startswith("I-"):
            legal_transition[prev_id, curr_id] = prev_tag in (f"B-{curr_tag[2:]}", f"I-{curr_tag[2:]}")

def bio_viterbi_decode(logits):
    emissions = logits.detach().float().cpu().numpy() if isinstance(logits, torch.Tensor) else np.asarray(logits, dtype=np.float64)
    if emissions.ndim != 2 or emissions.shape[1] != NUM_LABELS:
        raise ValueError(f"Expected [sequence_length, {NUM_LABELS}] emissions, got {emissions.shape}")
    if len(emissions) == 0:
        return []
    emissions = emissions.astype(np.float64, copy=False)
    neg_inf = -1e12
    scores = np.full_like(emissions, neg_inf, dtype=np.float64)
    back = np.zeros(emissions.shape, dtype=np.int32)
    scores[0, legal_start] = emissions[0, legal_start]
    for t in range(1, len(emissions)):
        cand = np.where(legal_transition, scores[t-1][:, None], neg_inf) + emissions[t][None, :]
        back[t] = np.argmax(cand, axis=0)
        scores[t] = cand[back[t], np.arange(NUM_LABELS)]
    last = int(np.argmax(scores[-1]))
    path = [last]
    for t in range(len(emissions)-1, 0, -1):
        path.append(int(back[t, path[-1]]))
    return list(reversed(path))

def decode_spans(text, offsets, pred_ids):
    spans, current_label, current_start, current_end = [], None, None, None
    def close():
        nonlocal current_label, current_start, current_end
        if current_label is not None:
            s, e = current_start, current_end
            while s < e and text[s].isspace(): s += 1
            while e > s and text[e-1].isspace(): e -= 1
            if s < e: spans.append((s, e, current_label))
        current_label = current_start = current_end = None
    for (s, e), pred_id in zip(offsets, pred_ids):
        if e <= s:
            continue
        tag = BIO_LABELS[int(pred_id)]
        if tag == "O":
            close(); continue
        prefix, lab = tag.split("-", 1)
        if prefix == "B" or current_label != lab:
            close(); current_label, current_start, current_end = lab, s, e
        else:
            current_end = e
    close()
    return sorted(set(spans))

DIAG_MAX_LEN, DIAG_STRIDE = 512, 448
@torch.inference_mode()
def diag_doc_probs(model, text):
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids, offsets = enc.ids, enc.offsets
    n = len(ids)
    if not n:
        return offsets, torch.zeros((0, len(BIO_LABELS)))
    logit_sum = torch.zeros((n, len(BIO_LABELS)), dtype=torch.float32)
    counts = torch.zeros(n, dtype=torch.float32)
    model.eval()
    for start in range(0, n, DIAG_STRIDE):
        end = min(start + DIAG_MAX_LEN, n)
        x = torch.tensor([ids[start:end]], dtype=torch.long, device=device)
        logits = model(x, torch.ones_like(x))[0].float().cpu()
        logit_sum[start:end] += logits
        counts[start:end] += 1
        if end == n: break
    avg_logits = logit_sum / counts.clamp_min(1).unsqueeze(-1)
    return offsets, torch.softmax(avg_logits, dim=-1)

def predict_texts(model, texts):
    result = {}
    model.eval()
    for doc_id, text in tqdm(texts.items(), desc="Predicting", leave=False):
        offsets, probs = diag_doc_probs(model, text)
        if not len(offsets):
            result[doc_id] = []
            continue
        pred_ids = torch.tensor(bio_viterbi_decode(torch.log(probs.clamp_min(1e-12))), dtype=torch.long)
        starts, ends = [a for a, b in offsets], [b for a, b in offsets]
        spans = []
        for s, e, lab in decode_spans(text, offsets, pred_ids.tolist()):
            ix = [i for i in range(bisect_right(ends, s), bisect_left(starts, e)) if ends[i] > starts[i]]
            conf = float(probs[ix, pred_ids[ix]].mean()) if ix else 0.0
            spans.append((int(s), int(e), lab, conf))
        result[doc_id] = spans
    return result

def apply_thr(raw, thresholds):
    return {d: [(s, e, lab) for s, e, lab, conf in spans if conf >= thresholds.get(lab, 0.0)] for d, spans in raw.items()}

def calculate_exact_span_metrics(gold_by_doc, pred_by_doc, labels=ENTITY_LABELS):
    metrics = {}
    for lab in labels:
        gold_set, pred_set = set(), set()
        for d, ents in gold_by_doc.items():
            for ent in ents:
                if len(ent) >= 3 and str(ent[2]) == lab:
                    gold_set.add((str(d), int(ent[0]), int(ent[1]), lab))
        for d, spans in pred_by_doc.items():
            for ent in spans:
                if len(ent) >= 3 and str(ent[2]) == lab:
                    pred_set.add((str(d), int(ent[0]), int(ent[1]), lab))
        tp = len(gold_set & pred_set); fp = len(pred_set - gold_set); fn = len(gold_set - pred_set)
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f = 2*p*r/(p+r) if p+r else 0.0
        metrics[lab] = {"gold":len(gold_set), "pred":len(pred_set), "TP":tp, "FP":fp, "FN":fn,
                        "precision":p, "recall":r, "f1":f}
    metrics["MACRO_F1"] = sum(metrics[x]["f1"] for x in labels) / len(labels)
    return metrics

# Verify exact recovery weights and a forward pass.
state = recovery["model_state_dict"]
loaded = champion_model.state_dict()
assert set(state) == set(loaded)
max_diff = max((loaded[k].detach().cpu() - state[k]).abs().max().item() for k in state)
assert max_diff == 0.0
with torch.inference_mode():
    z = champion_model(torch.zeros((1, 8), dtype=torch.long, device=device), torch.ones((1, 8), dtype=torch.long, device=device))
assert tuple(z.shape) == (1, 8, len(BIO_LABELS))
print("Champion weights verified exactly; output shape:", tuple(z.shape))

In [ ]:

# CS781: candidate rules for RESP headers and explicit witness identifiers.
# Validation is for evaluation only. No model or file is modified.

import re
from collections import Counter

RESP_LABEL = "RESP"
WIT_LABEL = "WIT"

# Matches common Indian judgment header styles:
# "... Versus NAME ... Respondent(s)"
# "RESPONDENT: NAME"
# "PETITIONER: NAME RESPONDENT: NAME"
RESP_MARKER = re.compile(
    r"\bRESPONDENTS?\b\s*[\(\[]?\s*[Ss]?\s*[\)\]]?\s*[:.]?",
    re.I
)
VERSUS_MARKER = re.compile(r"\b(?:VERSUS|VS\.?|V\.?)\b", re.I)

# Witness / listed-witness identifiers, including LW.
WIT_ID = re.compile(
    r"\b(?:P\s*\.?\s*W|D\s*\.?\s*W|C\s*\.?\s*W|L\s*\.?\s*W)"
    r"\s*[-.]?\s*\d+[A-Z]?\b",
    re.I
)

# Conservative name candidate: 1–5 capitalized name tokens, optionally
# including initials and common prefixes/titles.
NAME_TOKEN = (
    r"(?:[A-Z][A-Za-z.'’\-]*|"
    r"[A-Z]\.)"
)
NAME_CANDIDATE = re.compile(
    rf"(?<![\w])(?:Dr\.?\s+|Shri\s+|Smt\.?\s+|Mr\.?\s+|Mrs\.?\s+)?"
    rf"{NAME_TOKEN}(?:\s+{NAME_TOKEN}){{0,4}}"
)

def _add_candidate(out, text, start, end, label):
    # Trim whitespace and common punctuation without losing source offsets.
    while start < end and (text[start].isspace() or text[start] in " \t:;,."):
        start += 1
    while end > start and (text[end-1].isspace() or text[end-1] in " \t:;,."):
        end -= 1
    if start < end:
        out.add((start, end, label))

def resp_header_candidates(text, header_limit=2500):
    """Candidate spans from respondent header regions only."""
    out = set()
    header = text[:min(len(text), header_limit)]

    # Pattern A: names in the text between Versus and the Respondent marker.
    for vm in VERSUS_MARKER.finditer(header):
        rm = RESP_MARKER.search(header, vm.end())
        if rm is None:
            continue
        # Avoid grabbing a large body section if no nearby respondent label.
        if rm.start() - vm.end() > 700:
            continue

        segment_start, segment_end = vm.end(), rm.start()
        segment = header[segment_start:segment_end]

        # Split on common party separators, preserving absolute offsets.
        chunks = re.split(r"\s+(?:AND|&)\s+|[,;]\s*(?=\d|\b[A-Z])",
                          segment, flags=re.I)
        cursor = 0
        for chunk in chunks:
            local = segment.find(chunk, cursor)
            if local < 0:
                continue
            cursor = local + len(chunk)
            # Only keep name-like chunks, not legal role labels.
            for match in NAME_CANDIDATE.finditer(chunk):
                candidate = match.group(0).strip()
                if candidate.casefold() in {
                    "versus", "appellant", "appellants", "respondent",
                    "respondents", "petitioner", "petitioners"
                }:
                    continue
                s = segment_start + local + match.start()
                e = segment_start + local + match.end()
                _add_candidate(out, text, s, e, RESP_LABEL)

    # Pattern B: explicit "RESPONDENT: NAME" style metadata.
    for rm in re.finditer(r"\bRESPONDENT\b\s*:\s*", header, re.I):
        start = rm.end()
        end = min(len(header), start + 180)
        segment = header[start:end]
        # Stop at the next obvious metadata field or judgment heading.
        stop = re.search(
            r"\b(?:DATE OF JUDGMENT|BENCH|JUDGMENT|J U D G M E N T)\b",
            segment, re.I
        )
        if stop:
            segment = segment[:stop.start()]
        for match in NAME_CANDIDATE.finditer(segment):
            candidate = match.group(0).strip()
            if candidate.casefold() in {"and", "or", "ors", "the"}:
                continue
            _add_candidate(out, text, start + match.start(),
                           start + match.end(), RESP_LABEL)

    return out

def witness_id_candidates(text):
    """Candidate person spans immediately adjacent to witness identifiers."""
    out = set()

    for wm in WIT_ID.finditer(text):
        # Most common form: "Jagdish (PW-5)" or "Dr. X-PW2".
        left_start = max(0, wm.start() - 100)
        left = text[left_start:wm.start()]

        # Restrict to text after a sentence/punctuation boundary when possible.
        boundary_positions = [
            p for p in (left.rfind("."), left.rfind(";"),
                        left.rfind(":"), left.rfind("\n"))
            if p >= 0
        ]
        if boundary_positions:
            left_start += max(boundary_positions) + 1
            left = text[left_start:wm.start()]

        matches = list(NAME_CANDIDATE.finditer(left))
        if matches:
            # Keep the closest name-like expression before the identifier.
            m = matches[-1]
            candidate = m.group(0).strip()
            if candidate.casefold() not in {
                "witness", "statement", "evidence", "examined",
                "cross examined", "and", "the"
            }:
                _add_candidate(out, text, left_start + m.start(),
                               left_start + m.end(), WIT_LABEL)

        # Also handle "PW-5: Jagdish" or "LW-3, NAME".
        right = text[wm.end():min(len(text), wm.end() + 100)]
        right_boundary = re.search(r"[.;\n]", right)
        if right_boundary:
            right = right[:right_boundary.start()]
        m = NAME_CANDIDATE.search(right)
        if m:
            _add_candidate(out, text, wm.end() + m.start(),
                           wm.end() + m.end(), WIT_LABEL)

    return out

def candidates_for_text(text):
    return resp_header_candidates(text) | witness_id_candidates(text)

val_rule_candidates = {
    str(doc_id): sorted(candidates_for_text(text))
    for doc_id, text in val_texts.items()
}
train_rule_candidates = {
    str(doc_id): sorted(candidates_for_text(text))
    for doc_id, text in train_texts.items()
}

print("Candidate generation complete.")
print("Validation documents:", len(val_rule_candidates))
print("Validation RESP candidates:",
      sum(1 for spans in val_rule_candidates.values()
          for _, _, label in spans if label == RESP_LABEL))
print("Validation WIT candidates:",
      sum(1 for spans in val_rule_candidates.values()
          for _, _, label in spans if label == WIT_LABEL))
print("These are candidate rules, not accepted predictions.")


In [ ]:

# CS781 — Restore original rules + evaluate WIT hybrid
# Validation-only. No training, checkpoint writes, or prediction-file writes.

import re
from collections import Counter

_required = [
    "train_texts", "train_gold", "val_texts", "val_gold",
    "champion_model", "champ_thr", "predict_texts", "apply_thr",
    "diag_doc_probs", "bio_viterbi_decode", "decode_spans",
    "calculate_exact_span_metrics", "ENTITY_LABELS",
    "val_rule_candidates",
]
_missing = [x for x in _required if x not in globals()]
if _missing:
    raise RuntimeError("Missing required objects: " + ", ".join(_missing))

# ------------------------------------------------------------
# 1. Original counsel candidate rules from the original notebook
# ------------------------------------------------------------
# Reuse the original function if it exists; otherwise restore its
# conservative section-based candidate extraction.

if "rule_spans" not in globals():
    CAP = r"[A-ZÀ-Ö][A-Za-zÀ-ÖØ-öø-ÿ'’\-]*"
    INIT = r"[A-Z]\.(?:[ \t]*[A-Z]\.)*"
    WORD = rf"(?:{INIT}|{CAP})"
    NAME2 = rf"{WORD}(?:[ \t]+{WORD}){{0,3}}"

    SECTION_RE = re.compile(
        r"\b(?:"
        r"counsel\s+for\s+(?:the\s+)?(?:appellant|respondent|petitioner|"
        r"applicant|complainant|accused|state|opposite\s+party)s?"
        r"|for\s+(?:the\s+)?(?:appellant|respondent|petitioner|applicant|"
        r"complainant|accused|state|opposite\s+party)s?"
        r"|appearing\s+for"
        r"|agent\s+for\s+(?:the\s+)?(?:appellant|respondent|petitioner|"
        r"applicant|complainant|accused|state|opposite\s+party)s?"
        r")\s*[:\-]?\s*",
        re.I
    )
    SECTION_STOP_RE = re.compile(
        r"\b(?:Hon['’]?ble|JUDGMENT|ORDER|CORAM|"
        r"the\s+following\s+order|judgment\s+was\s+delivered)\b",
        re.I
    )

    def rule_spans(text):
        spans = set()
        for sm in SECTION_RE.finditer(text):
            start = sm.end()
            region = text[start:min(len(text), start + 500)]
            stop = SECTION_STOP_RE.search(region)
            if stop:
                region = region[:stop.start()]
            region = re.split(
                r"\n\s*\n|(?:\s{3,})|\b(?:versus|JUDGMENT|ORDER)\b",
                region, maxsplit=1, flags=re.I
            )[0]

            for m in re.finditer(NAME2, region):
                candidate = m.group(0).strip()
                if candidate.lower() in {
                    "appellant", "respondent", "petitioner", "applicant",
                    "state", "court", "justice", "judge", "counsel",
                    "advocate", "senior counsel", "learned counsel"
                }:
                    continue
                spans.add((start + m.start(), start + m.end()))
        return spans

    print(
        "NOTE: rule_spans was absent. Restored the section-based candidate "
        "extractor from the original notebook's candidate-rule family."
    )

# ------------------------------------------------------------
# 2. Original counsel side assignment and fold-1 policy selection
# ------------------------------------------------------------
PARTY_A = (
    r"(?:petitioners?|appellants?|applicants?|plaintiffs?|"
    r"complainants?|prosecution|revisionists?)"
)
PARTY_R = (
    r"(?:respondents?|defendants?|accused|non-\s*applicants?|"
    r"opposite\s+part(?:y|ies))"
)
FOR = r"(?:for|on\s+behalf\s+of|appearing\s+for|representing)"
MODS = r"(?:(?:learned|first|second|third|\d+\w*|all|other)\s+)*"
CUE_A = re.compile(rf"\b{FOR}\s+(?:the\s+)?{MODS}{PARTY_A}", re.I)
CUE_R = re.compile(rf"\b{FOR}\s+(?:the\s+)?{MODS}{PARTY_R}", re.I)
HDR_A = re.compile(rf"{PARTY_A}s?[\s.\-:]*(?:through|by)\s*[:\-]?\s*$", re.I)
HDR_R = re.compile(rf"{PARTY_R}s?[\s.\-:]*(?:through|by)\s*[:\-]?\s*$", re.I)
HON_TAIL = re.compile(r"(?:Mr|Ms|Mrs|Shri|Sri|Smt)\.?[ \t]*$")
A_, R_ = "A.COUNSEL", "R.COUNSEL"
COUNSEL_ORDER = ["hdr", "right", "left", "inherit"]

def _cues(text, lo, hi):
    return (
        [(m.start(), m.end(), A_) for m in CUE_A.finditer(text, lo, hi)]
        + [(m.start(), m.end(), R_) for m in CUE_R.finditer(text, lo, hi)]
    )

def _assign_side(text, s, e, prev):
    for step in COUNSEL_ORDER:
        if step == "right":
            cues = _cues(text, e, min(len(text), e + 200))
            if cues:
                return min(cues)[2], "right"
        elif step == "left":
            cues = _cues(text, max(0, s - 300), s)
            cues = [x for x in cues if x[1] <= s]
            if cues:
                return max(cues, key=lambda x: x[1])[2], "left"
        elif step == "hdr":
            tail = HON_TAIL.sub("", text[max(0, s - 80):s])
            if HDR_A.search(tail):
                return A_, "hdr"
            if HDR_R.search(tail):
                return R_, "hdr"
        elif step == "inherit":
            if prev and s - prev[0] <= 200 and prev[2] != "default":
                return prev[1], "inherit"
    return R_, "default"

def counsel_spans(text, policy):
    out, prev = [], None
    for s, e in sorted(rule_spans(text)):
        lab, source = _assign_side(text, s, e, prev)
        prev = (e, lab, source)
        if source == "default":
            if policy == "drop":
                continue
            if policy == "A":
                lab = A_
        out.append((int(s), int(e), lab))
    return out

def _counsel_fold1_score(policy):
    gold = {A_: set(), R_: set()}
    pred = {A_: set(), R_: set()}
    for d, ents in train_gold.items():
        for ent in ents:
            if len(ent) >= 3 and ent[2] in gold:
                gold[ent[2]].add((str(d), int(ent[0]), int(ent[1])))
    for d, text in train_texts.items():
        for s, e, lab in counsel_spans(text, policy):
            pred[lab].add((str(d), s, e))
    result = {}
    for lab in (A_, R_):
        tp = len(gold[lab] & pred[lab])
        p = tp / len(pred[lab]) if pred[lab] else 0.0
        r = tp / len(gold[lab]) if gold[lab] else 0.0
        result[lab] = 2*p*r/(p+r) if p+r else 0.0
    return result

_policy_scores = {
    p: _counsel_fold1_score(p) for p in ("R", "A", "drop")
}
chosen = max(
    _policy_scores,
    key=lambda p: (
        _policy_scores[p][A_] + _policy_scores[p][R_],
        p == "R"
    )
)
print("\nFold-1 counsel policy scores:")
for p, vals in _policy_scores.items():
    print(p, vals)
print("Selected counsel policy:", chosen)

# ------------------------------------------------------------
# 3. Original judge-pattern family and fold-1-only selection
# ------------------------------------------------------------
CAP = r"[A-ZÀ-Ö][A-Za-zÀ-ÖØ-öø-ÿ'’\-]*"
INIT = r"[A-Z]\.(?:[ \t]*[A-Z]\.)*"
WORD = rf"(?:{INIT}|{CAP})"
NAME2 = rf"{WORD}(?:[ \t]+{WORD}){{0,3}}"
TOK_RE = re.compile(WORD)

AJ_RE = re.compile(rf"(?i:\bjustice)[ \t]+({NAME2})")
B_RE = re.compile(r",[ \t]*J{1,2}\b\.?")
B2_RE = re.compile(
    r"[ \t]J{1,2}\.?(?=[ \t]*(?:in\b|speaking|writing|observed|"
    r"held\b|was\b|said\b|while\b|who\b|delivered|dissent|"
    r"agreed|also\b|noted|stated))"
)
C_RE = re.compile(r"\bJJ?\.?[ \t]*\(([^()\n]{3,45})\)")
D_RE = re.compile(
    r"(?:BENCH\s*:|\(J\)|\(J\.\))\s*"
    r"([A-Z][A-Za-z']+,[ \t]*[A-Z](?:[ \t]*\.?[ \t]*[A-Z])*\.?)"
    r"[ \t]*\(J\)"
)
E_RE = re.compile(
    r"\bJJ?\.?[ \t]*\[[ \t]*([A-Za-zÀ-ÿ\.'’ \t\-]{3,45}?)[ \t]*\]"
)
F_RE = re.compile(
    r"BENCH[ \t]*:[ \t]*([^\n]{3,120}?)"
    r"(?=[ \t]*(?:JUDG(?:E)?MENT|CITATION|ACT\b|\n|$))"
)
JUDGE_NAMES = [
    "A Justice NAME", "B NAME, J.", "B2 NAME J (no comma)",
    "C J.(NAME)", "D BENCH: X, Y (J)", "E J [NAME]", "F BENCH: A & B"
]
_STOP = {
    "JUDGMENT", "JUDGEMENT", "ORDER", "BENCH", "CORAM", "BEFORE",
    "PRESENT", "HON'BLE", "MR", "MRS", "MS", "JUSTICE", "THE",
    "AND", "OF", "DATE", "NEW", "DELHI", "BY", "DELIVERED"
}

def _trim_judge(text, s, e):
    while s < e and (text[s].isspace() or text[s] in ".,;:"):
        s += 1
    while e > s and (text[e-1].isspace() or text[e-1] in ",;:"):
        e -= 1
    return (s, e) if e > s else None

def _back_name(text, end, max_toks=4):
    ws = max(0, end - 80)
    toks = [(m.start()+ws, m.end()+ws) for m in TOK_RE.finditer(text[ws:end])]
    run = []
    for a, b in reversed(toks):
        nxt = run[0][0] if run else end
        if text[b:nxt].strip(" \t") or text[a:b].upper().strip(".") in _STOP:
            break
        run.insert(0, (a, b))
        if len(run) >= max_toks:
            break
    return (run[0][0], run[-1][1]) if run else None

def judge_patterns(text):
    out = {n: set() for n in JUDGE_NAMES}
    def add(n, s, e):
        span = _trim_judge(text, s, e)
        if span:
            out[n].add(span)
    for m in AJ_RE.finditer(text):
        add(JUDGE_NAMES[0], m.start(1), m.end(1))
    for rx, n in ((B_RE, JUDGE_NAMES[1]), (B2_RE, JUDGE_NAMES[2])):
        for m in rx.finditer(text):
            span = _back_name(text, m.start())
            if span:
                add(n, *span)
    for rx, n in ((C_RE, JUDGE_NAMES[3]), (E_RE, JUDGE_NAMES[5])):
        for m in rx.finditer(text):
            inner = m.group(1).strip()
            if inner and inner[0].isupper() and inner.upper() not in {"J", "JJ"} \
                    and re.fullmatch(r"[A-Za-zÀ-ÿ\.'’, \-]+", inner):
                add(n, m.start(1), m.end(1))
    for m in D_RE.finditer(text):
        add(JUDGE_NAMES[4], m.start(1), m.end(1))
    for m in F_RE.finditer(text):
        seg, base, pos = m.group(1), m.start(1), 0
        for part in re.split(r"[ \t]*(?:&|\bAND\b)[ \t]*", seg):
            k = seg.find(part, pos)
            if k < 0:
                continue
            pos = k + len(part)
            part2 = re.sub(r"[ \t]*\((?:C?J|J\.?)\)\s*$", "", part)
            span = _trim_judge(text, base+k, base+k+len(part2))
            if span and re.fullmatch(NAME2, text[span[0]:span[1]]):
                out[JUDGE_NAMES[6]].add(span)
    return out

def _resolve_judges(spans):
    out = []
    for s, e in sorted(set(spans), key=lambda x: (x[0], -(x[1]-x[0]))):
        if out and s < out[-1][1]:
            continue
        out.append((s, e))
    return out

_gold_judge = {
    (str(d), int(ent[0]), int(ent[1]))
    for d, ents in train_gold.items()
    for ent in ents if len(ent) >= 3 and str(ent[2]) == "JUDGE"
}
_pattern_preds = {n: set() for n in JUDGE_NAMES}
for d, text in train_texts.items():
    for name, spans in judge_patterns(text).items():
        _pattern_preds[name].update((str(d), s, e) for s, e in spans)

keep = []
for name in JUDGE_NAMES:
    pred = _pattern_preds[name]
    tp = len(pred & _gold_judge)
    precision = tp / len(pred) if pred else 0.0
    if precision >= 0.40 and tp >= 5:
        keep.append(name)

print("\nFold-1 selected judge patterns:", keep)

def judge_spans_doc(text, selected):
    candidates = set()
    for name, spans in judge_patterns(text).items():
        if name in selected:
            candidates.update(spans)
    return _resolve_judges(candidates)

# ------------------------------------------------------------
# 4. Rebuild original merge behavior, evaluate baseline, then WIT
# ------------------------------------------------------------
def merge_all(texts, champ_filtered, selected_judges):
    result = {}
    for d, text in texts.items():
        rules = list(counsel_spans(text, chosen)) + [
            (s, e, "JUDGE")
            for s, e in judge_spans_doc(text, selected_judges)
        ]
        kept = [
            span for span in champ_filtered.get(d, [])
            if span[2] not in (A_, R_, "JUDGE")
        ]
        kept = [
            span for span in kept
            if not any(span[0] < re_ and span[1] > rs
                       for rs, re_, _ in rules)
        ]
        result[d] = sorted(set(kept + rules))
    return result

In [ ]:

# CS781 — Original honorific-based counsel rules + V5 side assignment
# Restored from cs781_champion_plus_rules_run_cells.ipynb
# Policy is selected on fold 1 only.

import re

HON_RE = re.compile(
    r"\b(?:Mr|Ms|Mrs|Shri|Sri|Smt)\b\.?[ \t]*(?=[A-Z])"
)
NAME_TOK = re.compile(r"[A-Z][A-Za-z\.']*")
WS = re.compile(r"[ \t]+")
SIDE_A = re.compile(
    r"petitioner|appellant|applicant|plaintiff|complainant|prosecution", re.I
)
SIDE_R = re.compile(r"respondent|defendant|accused", re.I)

def rule_spans(text):
    out = []
    for m in HON_RE.finditer(text):
        pos = m.end()
        if text[pos:pos + 7] == "Justice":
            continue
        p, name_end = pos, None
        while True:
            t = NAME_TOK.match(text, p)
            if not t:
                break
            name_end = t.end()
            w = WS.match(text, name_end)
            if not w or not text[w.end():w.end() + 1].isupper():
                break
            p = w.end()
        if name_end is None:
            continue
        e = name_end
        while e > pos and text[e - 1] in ".,;:'":
            e -= 1
        if e > pos:
            out.append((pos, e))
    return out

PARTY_A = (
    r"(?:petitioners?|appellants?|applicants?|plaintiffs?|"
    r"complainants?|prosecution|revisionists?)"
)
PARTY_R = (
    r"(?:respondents?|defendants?|accused|non-\s*applicants?|"
    r"opposite\s+part(?:y|ies))"
)
FOR = r"(?:for|on\s+behalf\s+of|appearing\s+for|representing)"
MODS = r"(?:(?:learned|first|second|third|\d+\w*|all|other)\s+)*"

CUE_A = re.compile(rf"\b{FOR}\s+(?:the\s+)?{MODS}{PARTY_A}", re.I)
CUE_R = re.compile(rf"\b{FOR}\s+(?:the\s+)?{MODS}{PARTY_R}", re.I)
HDR_A = re.compile(rf"{PARTY_A}s?[\s.\-:]*(?:through|by)\s*[:\-]?\s*$", re.I)
HDR_R = re.compile(rf"{PARTY_R}s?[\s.\-:]*(?:through|by)\s*[:\-]?\s*$", re.I)
HON_TAIL = re.compile(r"(?:Mr|Ms|Mrs|Shri|Sri|Smt)\.?[ \t]*$")
A_, R_ = "A.COUNSEL", "R.COUNSEL"
ORDER = ["hdr", "right", "left", "inherit"]

def _cues(text, lo, hi):
    out = [(m.start(), m.end(), A_) for m in CUE_A.finditer(text, lo, hi)]
    out += [(m.start(), m.end(), R_) for m in CUE_R.finditer(text, lo, hi)]
    return out

def assign_side(text, s, e, order, prev, W_L=300, W_R=200):
    for step in order:
        if step == "right":
            cues = _cues(text, e, min(len(text), e + W_R))
            if cues:
                return min(cues)[2], "right"
        elif step == "left":
            cues = _cues(text, max(0, s - W_L), s)
            cues = [x for x in cues if x[1] <= s]
            if cues:
                return max(cues, key=lambda x: x[1])[2], "left"
        elif step == "hdr":
            tail = HON_TAIL.sub("", text[max(0, s - 80):s])
            if HDR_A.search(tail):
                return A_, "hdr"
            if HDR_R.search(tail):
                return R_, "hdr"
        elif step == "inherit":
            if prev and s - prev[0] <= 200 and prev[2] != "default":
                return prev[1], "inherit"
    return R_, "default"

def counsel_spans(text, policy):
    out, prev = [], None
    for s, e in sorted(rule_spans(text)):
        lab, src = assign_side(text, s, e, ORDER, prev)
        prev = (e, lab, src)
        if src == "default":
            if policy == "drop":
                continue
            if policy == "A":
                lab = A_
        out.append((int(s), int(e), lab))
    return out

def counsel_f1(texts, gold, policy):
    G = {A_: set(), R_: set()}
    P = {A_: set(), R_: set()}
    for d, ents in gold.items():
        for ent in ents:
            if len(ent) >= 3 and ent[2] in G:
                G[ent[2]].add((str(d), int(ent[0]), int(ent[1])))
    for d, text in texts.items():
        for s, e, lab in counsel_spans(text, policy):
            P[lab].add((str(d), s, e))
    result = {}
    for lab in (A_, R_):
        tp = len(G[lab] & P[lab])
        p = tp / len(P[lab]) if P[lab] else 0.0
        r = tp / len(G[lab]) if G[lab] else 0.0
        result[lab] = 2*p*r/(p+r) if p+r else 0.0
    return result

fold1_policy_scores = {
    policy: counsel_f1(train_texts, train_gold, policy)
    for policy in ("R", "A", "drop")
}
chosen = max(
    fold1_policy_scores,
    key=lambda p: (
        fold1_policy_scores[p][A_] + fold1_policy_scores[p][R_],
        p == "R"
    )
)

print("Fold-1-only counsel policy selection:")
for policy, vals in fold1_policy_scores.items():
    print(
        f"{policy:5s} A={vals[A_]:.4f} R={vals[R_]:.4f} "
        f"mean={(vals[A_] + vals[R_])/2:.4f}"
    )
print("Selected policy:", chosen)
print("Counsel rule functions restored.")


In [ ]:

# CS781 — Original fold-1 judge-pattern selection + verified WIT hybrid
# No model training and no checkpoint/prediction files are written.

import re

CAP = r"[A-ZÀ-Ö][A-Za-zÀ-ÖØ-öø-ÿ'’\-]*"
INIT = r"[A-Z]\.(?:[ \t]*[A-Z]\.)*"
WORD = rf"(?:{INIT}|{CAP})"
NAME2 = rf"{WORD}(?:[ \t]+{WORD}){{0,3}}"
TOK_RE = re.compile(WORD)

AJ_RE = re.compile(rf"(?i:\bjustice)[ \t]+({NAME2})")
B_RE = re.compile(r",[ \t]*J{1,2}\b\.?")
B2_RE = re.compile(
    r"[ \t]J{1,2}\.?(?=[ \t]*(?:in\b|speaking|writing|observed|"
    r"held\b|was\b|said\b|while\b|who\b|delivered|dissent|"
    r"agreed|also\b|noted|stated))"
)
C_RE = re.compile(r"\bJJ?\.?[ \t]*\(([^()\n]{3,45})\)")
D_RE = re.compile(
    r"(?:BENCH\s*:|\(J\)|\(J\.\))\s*"
    r"([A-Z][A-Za-z']+,[ \t]*[A-Z](?:[ \t]*\.?[ \t]*[A-Z])*\.?)"
    r"[ \t]*\(J\)"
)
E_RE = re.compile(
    r"\bJJ?\.?[ \t]*\[[ \t]*([A-Za-zÀ-ÿ\.'’ \t\-]{3,45}?)[ \t]*\]"
)
F_RE = re.compile(
    r"BENCH[ \t]*:[ \t]*([^\n]{3,120}?)"
    r"(?=[ \t]*(?:JUDG(?:E)?MENT|CITATION|ACT\b|\n|$))"
)
JUDGE_NAMES = [
    "A Justice NAME", "B NAME, J.", "B2 NAME J (no comma)",
    "C J.(NAME)", "D BENCH: X, Y (J)", "E J [NAME]", "F BENCH: A & B"
]
STOP_WORDS = {
    "JUDGMENT", "JUDGEMENT", "ORDER", "BENCH", "CORAM", "BEFORE",
    "PRESENT", "HON'BLE", "MR", "MRS", "MS", "JUSTICE", "THE",
    "AND", "OF", "DATE", "NEW", "DELHI", "BY", "DELIVERED"
}

def trim(text, s, e):
    while s < e and (text[s].isspace() or text[s] in ".,;:"):
        s += 1
    while e > s and (text[e-1].isspace() or text[e-1] in ",;:"):
        e -= 1
    return (s, e) if e > s else None

def back_name(text, end, max_toks=4):
    ws = max(0, end - 80)
    toks = [
        (m.start() + ws, m.end() + ws)
        for m in TOK_RE.finditer(text[ws:end])
    ]
    run = []
    for a, b in reversed(toks):
        nxt = run[0][0] if run else end
        if text[b:nxt].strip(" \t"):
            break
        if text[a:b].upper().strip(".") in STOP_WORDS:
            break
        run.insert(0, (a, b))
        if len(run) == max_toks:
            break
    return (run[0][0], run[-1][1]) if run else None

def judge_patterns2(text):
    out = {name: set() for name in JUDGE_NAMES}

    def add(name, s, e):
        span = trim(text, s, e)
        if span:
            out[name].add(span)

    for m in AJ_RE.finditer(text):
        add(JUDGE_NAMES[0], m.start(1), m.end(1))

    for rx, name in ((B_RE, JUDGE_NAMES[1]), (B2_RE, JUDGE_NAMES[2])):
        for m in rx.finditer(text):
            span = back_name(text, m.start())
            if span:
                add(name, *span)

    for rx, name in ((C_RE, JUDGE_NAMES[3]), (E_RE, JUDGE_NAMES[5])):
        for m in rx.finditer(text):
            inner = m.group(1).strip()
            if (
                inner and inner[0].isupper()
                and inner.upper() not in {"J", "JJ"}
                and re.fullmatch(r"[A-Za-zÀ-ÿ\.'’, \-]+", inner)
            ):
                add(name, m.start(1), m.end(1))

    for m in D_RE.finditer(text):
        add(JUDGE_NAMES[4], m.start(1), m.end(1))

    for m in F_RE.finditer(text):
        segment, base, pos = m.group(1), m.start(1), 0
        for part in re.split(r"[ \t]*(?:&|\bAND\b)[ \t]*", segment):
            k = segment.find(part, pos)
            if k < 0:
                continue
            pos = k + len(part)
            part2 = re.sub(
                r"[ \t]*\((?:C?J|J\.?)\)\s*$", "", part
            )
            span = trim(text, base + k, base + k + len(part2))
            if span and re.fullmatch(NAME2, text[span[0]:span[1]]):
                out[JUDGE_NAMES[6]].add(span)

    return out

def resolve_judge_overlaps(spans):
    out = []
    for s, e in sorted(set(spans), key=lambda x: (x[0], -(x[1]-x[0]))):
        if out and s < out[-1][1]:
            continue
        out.append((s, e))
    return out

# Select patterns on fold 1 only.
gold_judge_train = {
    (str(d), int(ent[0]), int(ent[1]))
    for d, ents in train_gold.items()
    for ent in ents
    if len(ent) >= 3 and str(ent[2]) == "JUDGE"
}
pattern_preds = {name: set() for name in JUDGE_NAMES}

for d, text in train_texts.items():
    for name, spans in judge_patterns2(text).items():
        pattern_preds[name].update(
            (str(d), s, e) for s, e in spans
        )

keep = []
print("Fold-1-only judge-pattern selection:")
for name in JUDGE_NAMES:
    pred = pattern_preds[name]
    tp = len(gold_judge_train & pred)
    precision = tp / len(pred) if pred else 0.0
    recall = tp / len(gold_judge_train) if gold_judge_train else 0.0
    print(
        f"{name:26s} predicted={len(pred):4d} TP={tp:3d} "
        f"P={precision:.3f} R={recall:.3f}"
    )
    if precision >= 0.40 and tp >= 5:
        keep.append(name)

print("Selected judge patterns:", keep)

def judge_spans_doc(text, selected):
    spans = set()
    for name, candidates in judge_patterns2(text).items():
        if name in selected:
            spans.update(candidates)
    return resolve_judge_overlaps(spans)

# Use the established champion thresholds, not finalv1's altered champ_thr.
ESTABLISHED_THR = {
    "APP": 0.15,
    "RESP": 0.45,
    "A.COUNSEL": 0.45,
    "R.COUNSEL": 0.45,
    "JUDGE": 0.25,
    "WIT": 0.45,
    "AUTH": 0.35,
    "COURT": 0.45,
    "STAT": 0.25,
    "PREC": 0.35,
    "DATE": 0.55,
    "CASENO": 0.35,
}

# Verify the inference helpers and candidate rules exist.
for name in (
    "champion_model", "predict_texts", "apply_thr",
    "calculate_exact_span_metrics", "ENTITY_LABELS",
    "val_rule_candidates", "val_texts", "val_gold"
):
    if name not in globals():
        raise RuntimeError(f"Missing required object: {name}")

# Build baseline: thresholded champion + original counsel + selected judges.
raw_val_original = predict_texts(champion_model, val_texts)
filtered_val_original = apply_thr(raw_val_original, ESTABLISHED_THR)

def merge_all_original(texts, champion_filtered, selected_judges):
    result = {}
    for d, text in texts.items():
        rules = list(counsel_spans(text, chosen))
        rules += [
            (s, e, "JUDGE")
            for s, e in judge_spans_doc(text, selected_judges)
        ]

        kept = [
            span for span in champion_filtered.get(d, [])
            if span[2] not in ("A.COUNSEL", "R.COUNSEL", "JUDGE")
        ]
        kept = [
            span for span in kept
            if not any(
                span[0] < rule_e and span[1] > rule_s
                for rule_s, rule_e, _ in rules
            )
        ]
        result[d] = sorted(set(kept + rules))
    return result

In [ ]:
# Self-contained WIT helper definitions for the new experiments.
# No baseline globals or previous evaluation cells are required.

import re


_WIT_ID = re.compile(
    r"\b(?:P\s*\.?\s*W|D\s*\.?\s*W|C\s*\.?\s*W|L\s*\.?\s*W)"
    r"\s*[-.]?\s*\d+[A-Z]?\b", re.I
)

# Terms seen in the error audit that are not person-name spans in context.
_NON_PERSON = {
    "it", "both", "pw", "dw", "cw", "lw", "a", "an", "the",
    "dr", "court", "superintendent", "ex-sarpanch",
    "witness", "evidence", "statement", "respondent", "appellant",
    "accused", "petitioner", "deceased", "complainant",
}

def _near_identifier(text, s, e, radius=60):
    region = text[max(0, s-radius):min(len(text), e+radius)]
    return bool(_WIT_ID.search(region))

def _candidate_surface(text, s, e):
    return text[s:e].strip(" \t\r\n,.;:()[]{}-–—")

def _valid_person_surface(surface, mode):
    normalized = surface.casefold().strip()
    words = re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ]+", surface)

    if normalized in _NON_PERSON:
        return False

    # Reject fragments that are just witness-ID tokens.
    if re.fullmatch(r"(?:P|D|C|L)?W\d*", normalized.replace("-", "")):
        return False

    if mode in ("non_person", "name_shape", "name_shape_plus_context"):
        # Require at least one alphabetic word of reasonable length.
        if not words or max(map(len, words)) < 3:
            return False

    if mode in ("name_shape", "name_shape_plus_context"):
        # A single token can be a legitimate name, so do not reject it
        # merely for having one word.
        if len(surface.strip()) < 4:
            return False

    return True

In [ ]:
# NEW EXPERIMENTS — weighted cross-entropy vs focal loss
# IMPORTANT: run this notebook top-to-bottom; this cell no longer depends on old result variables.
# Both runs start independently from the champion checkpoint.
# Fold 1 trains; fold 2 selects/evaluates. Test labels are never used.
# Existing champion and predictions are not overwritten.

import copy, gc, json, math
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

SEED = 78115
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

VERIFIED_THR = {
    "APP": 0.15, "RESP": 0.45, "A.COUNSEL": 0.45, "R.COUNSEL": 0.45,
    "JUDGE": 0.25, "WIT": 0.45, "AUTH": 0.35, "COURT": 0.45,
    "STAT": 0.25, "PREC": 0.35, "DATE": 0.55, "CASENO": 0.35,
}
REFERENCE_SCORE = 0.333307
WIT_EXCLUSIONS = {
    "associate professor", "both pw", "manchili",
    "oblique", "subsequently", "the police patil",
}
TRIM_CHARS = " \t\r\n,.;:()[]{}"

# ---------- Build exactly the same validated hybrid pipeline ----------
def exp_full_pipeline(model, texts, gold=None):
    model.eval()
    raw = predict_texts(model, texts)
    thresholded = apply_thr(raw, VERIFIED_THR)
    base = merge_all_original(texts, thresholded, keep)

    result = {
        str(d): set((int(s), int(e), str(lab)) for s, e, lab in spans)
        for d, spans in base.items()
    }

    # Use the exact WIT candidate generator and validated filters.
    candidates_by_doc = (
        val_rule_candidates if texts is val_texts else
        {str(d): sorted(candidates_for_text(t)) for d, t in texts.items()}
    )
    for did, candidates in candidates_by_doc.items():
        did = str(did)
        if did not in texts:
            continue
        text = texts[did]
        pred = result.setdefault(did, set())

        for s, e, label in candidates:
            if str(label) != "WIT":
                continue
            s, e = int(s), int(e)
            surface = _candidate_surface(text, s, e)
            if not _near_identifier(text, s, e, radius=60):
                continue
            if not _valid_person_surface(surface, "name_shape_plus_context"):
                continue
            if not _near_identifier(text, s, e, radius=30):
                continue
            if surface.casefold() in WIT_EXCLUSIONS:
                continue
            if any(max(s, ps) < min(e, pe) for ps, pe, _ in pred):
                continue
            pred.add((s, e, "WIT"))

    # Best validated boundary configuration: trim CASENO + STAT only.
    final = {}
    for did, spans in result.items():
        text = texts[did]
        cleaned = set()
        for s, e, lab in spans:
            if lab in ("CASENO", "STAT"):
                while s < e and text[s] in TRIM_CHARS:
                    s += 1
                while e > s and text[e-1] in TRIM_CHARS:
                    e -= 1
            if s < e:
                cleaned.add((int(s), int(e), str(lab)))
        final[did] = sorted(cleaned)

    if gold is None:
        return final
    return final, calculate_exact_span_metrics(gold, final, ENTITY_LABELS)

print("=== Rebuild reference hybrid before training ===")
champion_model.eval()
reference_predictions, reference_metrics = exp_full_pipeline(
    champion_model, val_texts, val_gold
)
reference_score = float(reference_metrics["MACRO_F1"])
print(f"Rebuilt score: {reference_score:.6f} | target: {REFERENCE_SCORE:.6f}")
if abs(reference_score - REFERENCE_SCORE) > 0.002:
    raise RuntimeError(
        "Reference pipeline mismatch; no experiment was trained. "
        "The rebuilt pipeline must match the known 0.333307 baseline."
    )

# ---------- Prepare fold-1 BIO training chunks ----------
MAX_LEN, STRIDE, BATCH_SIZE = 512, 448, 1
EPOCHS = 1
ENCODER_LR = 5e-6
CLASSIFIER_LR = 2e-5

def encode_train_doc(text, entities):
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids, offsets = list(enc.ids), list(enc.offsets)
    tags = [label2id["O"]] * len(ids)
    conflicts = [False] * len(ids)

    for ent in entities:
        if len(ent) < 3:
            continue
        s, e, lab = int(ent[0]), int(ent[1]), str(ent[2])
        if lab not in ENTITY_LABELS or s < 0 or e <= s or e > len(text):
            continue
        tok_ix = [
            i for i, (a, b) in enumerate(offsets)
            if b > a and a < e and b > s
        ]
        if not tok_ix:
            continue
        for j, i in enumerate(tok_ix):
            tag_id = label2id[("B-" if j == 0 else "I-") + lab]
            if tags[i] != label2id["O"] and tags[i] != tag_id:
                conflicts[i] = True
            else:
                tags[i] = tag_id

    for i, bad in enumerate(conflicts):
        if bad:
            tags[i] = -100

    chunks, seen = [], set()
    for start in range(0, len(ids), STRIDE):
        end = min(start + MAX_LEN, len(ids))
        if start >= end:
            break
        chunk_labels = []
        for i in range(start, end):
            if i in seen:
                chunk_labels.append(-100)
            else:
                chunk_labels.append(tags[i])
                seen.add(i)
        chunks.append({"input_ids": ids[start:end], "labels": chunk_labels})
        if end == len(ids):
            break
    return chunks, tags

train_chunks = []
tag_counts = torch.zeros(len(BIO_LABELS), dtype=torch.long)
for did, text in train_texts.items():
    chunks, full_tags = encode_train_doc(text, train_gold[did])
    train_chunks.extend(chunks)
    for tag in full_tags:
        if tag >= 0:
            tag_counts[tag] += 1

assert train_chunks
print("Training docs:", len(train_texts), "| chunks:", len(train_chunks))
print("Validation docs:", len(val_texts), "| test docs:", len(test_texts))

def collate_train(batch):
    n = max(len(x["input_ids"]) for x in batch)
    ids = torch.zeros((len(batch), n), dtype=torch.long)
    mask = torch.zeros_like(ids)
    labels = torch.full((len(batch), n), -100, dtype=torch.long)
    for i, item in enumerate(batch):
        k = len(item["input_ids"])
        ids[i, :k] = torch.tensor(item["input_ids"], dtype=torch.long)
        mask[i, :k] = 1
        labels[i, :k] = torch.tensor(item["labels"], dtype=torch.long)
    return ids, mask, labels

train_loader = DataLoader(
    train_chunks, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_train
)

# New experiment A: bounded inverse-frequency weighted cross entropy.
weights = tag_counts.float().clamp_min(1).pow(-0.25)
weights = (weights / weights.mean()).clamp(0.5, 3.0).to(device)

# New experiment B: focal loss (gamma=2) with the same bounded class alpha.
def focal_token_loss(logits, labels, alpha, gamma=2.0):
    flat_logits = logits.reshape(-1, logits.shape[-1])
    flat_labels = labels.reshape(-1)
    valid = flat_labels != -100
    if not valid.any():
        return flat_logits.sum() * 0.0
    logits = flat_logits[valid]
    labels = flat_labels[valid]
    log_probs = torch.log_softmax(logits, dim=-1)
    log_pt = log_probs.gather(1, labels[:, None]).squeeze(1)
    pt = log_pt.exp()
    a = alpha[labels]
    return (-a * ((1.0 - pt).clamp_min(0) ** gamma) * log_pt).mean()

def run_training_variant(name, loss_kind):
    # Independent model copy from the same untouched champion.
    model = copy.deepcopy(champion_model).to(device)
    model.train()
    optimizer = torch.optim.AdamW(
        [
            {"params": model.encoder.parameters(), "lr": ENCODER_LR},
            {"params": model.classifier.parameters(), "lr": CLASSIFIER_LR},
        ],
        weight_decay=0.01,
    )
    ce_loss = nn.CrossEntropyLoss(weight=weights, ignore_index=-100)
    losses = []

    for epoch in range(EPOCHS):
        model.train()
        for ids, mask, labels in tqdm(
            train_loader, desc=f"{name} epoch {epoch+1}/{EPOCHS}"
        ):
            ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(ids, mask)
            if loss_kind == "weighted_ce":
                loss = ce_loss(
                    logits.reshape(-1, len(BIO_LABELS)), labels.reshape(-1)
                )
            elif loss_kind == "focal":
                loss = focal_token_loss(logits, labels, weights, gamma=2.0)
            else:
                raise ValueError(loss_kind)

            if not torch.isfinite(loss):
                raise RuntimeError(f"{name}: non-finite loss.")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            losses.append(float(loss.item()))

    model.eval()
    predictions, metrics = exp_full_pipeline(model, val_texts, val_gold)
    score = float(metrics["MACRO_F1"])
    print(f"\n{name}: mean loss={np.mean(losses):.5f}, macro-F1={score:.6f}, "
          f"delta={score-reference_score:+.6f}")
    for lab in ENTITY_LABELS:
        old, new = reference_metrics[lab]["f1"], metrics[lab]["f1"]
        print(f"  {lab:12s} {old:.4f} -> {new:.4f} ({new-old:+.4f})")

    return {
        "name": name,
        "score": score,
        "metrics": metrics,
        "predictions": predictions,
        "model": model,
    }

# Run two distinct, one-epoch experiments from the exact same starting checkpoint.
expA = run_training_variant("Experiment A — weighted cross-entropy", "weighted_ce")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

expB = run_training_variant("Experiment B — focal loss (gamma=2)", "focal")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

results = sorted([expA, expB], key=lambda x: x["score"], reverse=True)
print("\n=== EXPERIMENT LEADERBOARD ===")
print(f"Reference hybrid: {reference_score:.6f}")
for r in results:
    print(f"{r['name']}: {r['score']:.6f} (delta {r['score']-reference_score:+.6f})")

# Save each experimental checkpoint only if it beats the known reference.
for r in results:
    if r["score"] > REFERENCE_SCORE:
        out = Path("/kaggle/working") / (
            "experiment15_weighted_ce.pt"
            if r["name"].endswith("weighted cross-entropy")
            else "experiment16_focal_loss.pt"
        )
        torch.save({
            "model_state_dict": r["model"].state_dict(),
            "bio_labels": BIO_LABELS,
            "experiment": r["name"],
            "validation_macro_f1": float(r["score"]),
            "reference_macro_f1": float(REFERENCE_SCORE),
            "epochs": EPOCHS,
        }, out)
        print("Saved improved candidate checkpoint:", out)

if all(r["score"] <= REFERENCE_SCORE for r in results):
    print("Neither new experiment beat the reference; champion remains unchanged.")

print("\nNo test gold labels were used.")
print("Champion weights and existing prediction files were not modified.")

In [ ]:

import random
import re
from collections import defaultdict, Counter

# Use a fixed seed so the generated examples are reproducible.
AUG_SEED = 781
_rng = random.Random(AUG_SEED)

# Expected formats:
# train_texts: {doc_id: full_document_text}
# train_gold:  {doc_id: [[start, end, label], ...]}
#
# This cell assumes character offsets use [start, end).

def _normalise_span(span):
    return int(span[0]), int(span[1]), str(span[2])

replacement_pool = defaultdict(set)
entity_counts = Counter()
span_errors = []

for doc_id, text in train_texts.items():
    annotations = train_gold[str(doc_id)] if str(doc_id) in train_gold else train_gold[doc_id]

    for raw_span in annotations:
        start, end, label = _normalise_span(raw_span)

        if not (0 <= start < end <= len(text)):
            span_errors.append((doc_id, start, end, label, "out_of_bounds"))
            continue

        surface = text[start:end]
        if not surface.strip():
            span_errors.append((doc_id, start, end, label, "empty_surface"))
            continue

        # Keep the exact surface, including punctuation/casing.
        replacement_pool[label].add(surface)
        entity_counts[label] += 1

if span_errors:
    print("Invalid training spans:", len(span_errors))
    print("First few:", span_errors[:10])
    raise ValueError("Fix invalid training annotations before augmentation.")

print("Training documents:", len(train_texts))
print("Training entities:", sum(entity_counts.values()))
print("Replacement candidates by label:")
for label in sorted(replacement_pool):
    print(f"{label:12s} {len(replacement_pool[label]):5d} unique surfaces")


In [ ]:

def make_entity_replacement_example(text, annotations, rng, max_replacements=2):
    """
    Replace a small number of annotated mentions with same-label
    surfaces from the training-only replacement pools.

    Returns (new_text, new_annotations).
    Annotations use [start, end, label], with end exclusive.
    """
    spans = sorted(
        [_normalise_span(s) for s in annotations],
        key=lambda x: (x[0], x[1])
    )

    # Verify bounds and non-overlap before modifying anything.
    previous_end = 0
    for start, end, label in spans:
        if not (0 <= start < end <= len(text)):
            raise ValueError(f"Invalid span: {(start, end, label)}")
        if start < previous_end:
            raise ValueError("Overlapping gold spans detected.")
        previous_end = end

    eligible = []
    for i, (start, end, label) in enumerate(spans):
        original_surface = text[start:end]
        options = [
            x for x in replacement_pool.get(label, set())
            if x != original_surface
            and x.strip()
            and "\n" not in x
            and "\r" not in x
        ]
        if options:
            eligible.append((i, options))

    if not eligible:
        return text, spans

    rng.shuffle(eligible)
    selected = {
        i: rng.choice(options)
        for i, options in eligible[:max_replacements]
    }

    pieces = []
    new_annotations = []
    cursor = 0
    new_length = 0

    for i, (start, end, label) in enumerate(spans):
        prefix = text[cursor:start]
        pieces.append(prefix)
        new_length += len(prefix)

        surface = selected.get(i, text[start:end])
        new_start = new_length
        pieces.append(surface)
        new_length += len(surface)
        new_end = new_length

        new_annotations.append([new_start, new_end, label])
        cursor = end

    pieces.append(text[cursor:])
    new_text = "".join(pieces)

    # Mandatory exact-offset integrity check.
    for start, end, label in new_annotations:
        assert 0 <= start < end <= len(new_text)
        assert new_text[start:end].strip()

    return new_text, new_annotations


# Smoke test on a single training document.
_smoke_id = next(iter(train_texts))
_smoke_text = train_texts[_smoke_id]
_smoke_gold = train_gold[str(_smoke_id)] if str(_smoke_id) in train_gold else train_gold[_smoke_id]

_smoke_aug_text, _smoke_aug_gold = make_entity_replacement_example(
    _smoke_text, _smoke_gold, random.Random(AUG_SEED)
)

assert len(_smoke_aug_gold) == len(_smoke_gold)
print("Smoke test passed.")
print("Original characters:", len(_smoke_text))
print("Augmented characters:", len(_smoke_aug_text))
print("Entity count preserved:", len(_smoke_aug_gold))


In [ ]:

aug_texts = dict(train_texts)
aug_gold = {
    doc_id: [
        list(_normalise_span(s))
        for s in (train_gold[str(doc_id)] if str(doc_id) in train_gold else train_gold[doc_id])
    ]
    for doc_id in train_texts
}

# One synthetic variant per original training document.
for doc_id, text in train_texts.items():
    gold = aug_gold[doc_id]
    new_text, new_gold = make_entity_replacement_example(
        text, gold, random.Random(f"{AUG_SEED}-{doc_id}"),
        max_replacements=2
    )

    # Add only examples that actually changed.
    if new_text != text:
        synthetic_id = f"{doc_id}__entity_replacement"
        aug_texts[synthetic_id] = new_text
        aug_gold[synthetic_id] = new_gold

assert set(train_texts).isdisjoint(
    set(aug_texts) - set(train_texts)
)

print("Original training documents:", len(train_texts))
print("Synthetic documents added:", len(aug_texts) - len(train_texts))
print("Total training documents:", len(aug_texts))


In [ ]:

# This cell intentionally does not train on any additional data.
# It verifies that the named training/validation/test pools are separate.

train_ids = set(map(str, train_texts.keys()))
val_ids = set(map(str, val_texts.keys()))

test_ids = set(map(str, test_texts.keys())) if "test_texts" in globals() else set()

assert train_ids.isdisjoint(val_ids), "Training/validation ID overlap!"
assert train_ids.isdisjoint(test_ids), "Training/test ID overlap!"
assert val_ids.isdisjoint(test_ids), "Validation/test ID overlap!"

print("Training documents:", len(train_ids))
print("Validation documents:", len(val_ids))
print("Test documents:", len(test_ids))
print()
print("No separate eligible unlabeled pool is declared by this cell.")
print("Only proceed with pseudo-labeling if you identify another official")
print("Phase 2 training-text pool and confirm its use is permitted.")


In [ ]:

from collections import Counter

def validate_character_annotations(texts, annotations_by_doc):
    errors = []
    checked_spans = 0

    for doc_id, text in texts.items():
        spans = annotations_by_doc.get(doc_id)

        if spans is None:
            errors.append((doc_id, "missing_annotations"))
            continue

        ordered = sorted(
            [(int(s), int(e), str(label)) for s, e, label in spans],
            key=lambda x: (x[0], x[1])
        )

        previous_end = 0
        seen = set()

        for start, end, label in ordered:
            checked_spans += 1

            if not (0 <= start < end <= len(text)):
                errors.append((doc_id, "out_of_bounds", start, end, label))
                continue

            if start < previous_end:
                errors.append((doc_id, "overlapping_spans", start, end, label))

            if (start, end, label) in seen:
                errors.append((doc_id, "duplicate_span", start, end, label))

            if not text[start:end].strip():
                errors.append((doc_id, "empty_surface", start, end, label))

            seen.add((start, end, label))
            previous_end = max(previous_end, end)

    print("Documents checked:", len(texts))
    print("Spans checked:", checked_spans)
    print("Errors:", len(errors))

    if errors:
        print("First 20 errors:")
        for err in errors[:20]:
            print(err)
        raise ValueError("Character annotation validation failed.")

    print("PASS: all character annotations are valid.")


validate_character_annotations(aug_texts, aug_gold)


In [ ]:

def span_triplet(span):
    """Normalize an annotation to (start, end, label), ignoring extra fields."""
    if len(span) < 3:
        raise ValueError(f"Annotation has fewer than 3 fields: {span!r}")
    return [int(span[0]), int(span[1]), str(span[2])]


# Verify that original examples and their labels remain unchanged.
for doc_id, original_text in train_texts.items():
    assert aug_texts[doc_id] == original_text, (
        f"Original text changed for {doc_id}"
    )

    original_gold = (
        train_gold[str(doc_id)]
        if str(doc_id) in train_gold
        else train_gold[doc_id]
    )

    expected = [span_triplet(s) for s in original_gold]
    actual = [span_triplet(s) for s in aug_gold[doc_id]]

    assert actual == expected, (
        f"Original annotations changed for {doc_id}"
    )

synthetic_ids = set(aug_texts) - set(train_texts)

print("Original training examples preserved.")
print("Synthetic examples:", len(synthetic_ids))
print(
    "Original documents with an augmented variant:",
    sum(aug_texts[d] != train_texts[d] for d in train_texts)
)


In [ ]:

shown = 0

for doc_id in sorted(synthetic_ids):
    original_id = doc_id.removesuffix("__entity_replacement")

    if original_id not in train_texts:
        continue

    original_text = train_texts[original_id]
    augmented_text = aug_texts[doc_id]

    print("=" * 80)
    print("Document:", original_id)
    print("Original excerpt:")
    print(original_text[:500])
    print("\nAugmented excerpt:")
    print(augmented_text[:500])

    shown += 1
    if shown >= 3:
        break

if shown == 0:
    print("No synthetic examples were generated.")


In [ ]:

# Diagnose why the generated variants did not change.

for doc_id in list(train_texts)[:5]:
    text = train_texts[doc_id]
    gold = (
        train_gold[str(doc_id)]
        if str(doc_id) in train_gold
        else train_gold[doc_id]
    )

    eligible_count = 0
    for span in gold:
        start, end, label = int(span[0]), int(span[1]), str(span[2])
        surface = text[start:end]
        options = [
            x for x in replacement_pool.get(label, set())
            if x != surface
            and x.strip()
            and "\n" not in x
            and "\r" not in x
        ]
        if options:
            eligible_count += 1

    test_text, test_gold = make_entity_replacement_example(
        text, gold, random.Random(f"diagnostic-{doc_id}"),
        max_replacements=2
    )

    print(
        doc_id,
        "| eligible spans:", eligible_count,
        "| changed:", test_text != text,
        "| original chars:", len(text),
        "| new chars:", len(test_text)
    )


In [ ]:

# Rebuild synthetic variants from the original training data.
# Preserve original examples and annotations unchanged.

aug_texts = dict(train_texts)
aug_gold = {}

for doc_id, text in train_texts.items():
    gold = (
        train_gold[str(doc_id)]
        if str(doc_id) in train_gold
        else train_gold[doc_id]
    )

    aug_gold[doc_id] = [
        [int(s[0]), int(s[1]), str(s[2])]
        for s in gold
    ]

changed_count = 0
unchanged_count = 0

for doc_id, text in train_texts.items():
    gold = (
        train_gold[str(doc_id)]
        if str(doc_id) in train_gold
        else train_gold[doc_id]
    )

    new_text, new_gold = make_entity_replacement_example(
        text,
        gold,
        random.Random(f"{AUG_SEED}-{doc_id}"),
        max_replacements=2
    )

    synthetic_id = f"{doc_id}__entity_replacement"

    if new_text != text:
        aug_texts[synthetic_id] = new_text
        aug_gold[synthetic_id] = new_gold
        changed_count += 1
    else:
        unchanged_count += 1

assert len(aug_texts) == len(train_texts) + changed_count
assert all(
    aug_texts[d] == train_texts[d]
    for d in train_texts
)

validate_character_annotations(aug_texts, aug_gold)

print("Original training documents:", len(train_texts))
print("Synthetic variants created:", changed_count)
print("Documents with no eligible replacement:", unchanged_count)
print("Total training examples:", len(aug_texts))


In [ ]:

import re
from collections import Counter, defaultdict

def surface_shape(text):
    text = text.strip()
    if not text:
        return "empty"
    if re.fullmatch(r"[\d\s./()-]+", text):
        return "numeric_or_case_id"
    if re.fullmatch(r"[A-Z][A-Z .'-]*", text):
        return "uppercase_name_or_heading"
    if re.search(r"\d", text):
        return "contains_digits"
    if text[0].isupper():
        return "capitalized"
    return "other"

# Match each synthetic span against the corresponding original span
# by annotation order. This is a review aid, not a semantic proof.
replacement_audit = []

for doc_id in train_texts:
    synthetic_id = f"{doc_id}__entity_replacement"
    if synthetic_id not in aug_texts:
        continue

    original_text = train_texts[doc_id]
    synthetic_text = aug_texts[synthetic_id]

    original_gold = (
        train_gold[str(doc_id)]
        if str(doc_id) in train_gold else train_gold[doc_id]
    )
    synthetic_gold = aug_gold[synthetic_id]

    if len(original_gold) != len(synthetic_gold):
        raise ValueError(f"Entity count changed: {doc_id}")

    for old, new in zip(original_gold, synthetic_gold):
        old_start, old_end, old_label = int(old[0]), int(old[1]), str(old[2])
        new_start, new_end, new_label = int(new[0]), int(new[1]), str(new[2])

        old_surface = original_text[old_start:old_end]
        new_surface = synthetic_text[new_start:new_end]

        if old_label != new_label:
            raise ValueError(f"Label changed for {doc_id}")

        if old_surface != new_surface:
            replacement_audit.append({
                "doc_id": doc_id,
                "label": old_label,
                "old": old_surface,
                "new": new_surface,
                "old_shape": surface_shape(old_surface),
                "new_shape": surface_shape(new_surface),
            })

print("Actual replacements:", len(replacement_audit))
print("Replacements by label:")
for label, count in sorted(Counter(x["label"] for x in replacement_audit).items()):
    print(f"{label:12s} {count:4d}")

print("\nFirst 30 replacements to review:")
for item in replacement_audit[:30]:
    print(
        f'{item["label"]:12s} | {item["old"]!r} -> {item["new"]!r} '
        f'| {item["old_shape"]} -> {item["new_shape"]}'
    )


In [ ]:

import inspect

keywords = (
    "token", "bio", "align", "encode", "chunk",
    "label", "offset", "dataset", "collat"
)

matches = []
for name, obj in list(globals().items()):
    if not callable(obj):
        continue
    if not any(k in name.lower() for k in keywords):
        continue
    try:
        signature = str(inspect.signature(obj))
    except Exception:
        signature = "(signature unavailable)"
    matches.append((name, signature))

print("Relevant functions:")
for name, signature in sorted(matches):
    print(f"{name}{signature}")


In [ ]:

if "tokenizer" not in globals():
    print("No `tokenizer` variable found; use the tokenizer identified in Cell 2.")
else:
    sample_id = next(iter(aug_texts))
    sample_text = aug_texts[sample_id]
    sample_encoding = tokenizer.encode(sample_text)

    ids = sample_encoding.ids
    offsets = sample_encoding.offsets

    assert len(ids) == len(offsets), "Token IDs and offsets differ in length."

    nonempty_offsets = [
        (s, e) for s, e in offsets
        if isinstance(s, int) and isinstance(e, int) and e > s
    ]

    print("Sample:", sample_id)
    print("Text characters:", len(sample_text))
    print("Tokens:", len(ids))
    print("Tokens with non-empty offsets:", len(nonempty_offsets))
    print("First 20 offsets:", offsets[:20])


In [ ]:

import inspect
import traceback

print("encode_train_doc signature:")
print(inspect.signature(encode_train_doc))
print("\nFunction source:")
print(inspect.getsource(encode_train_doc))


In [ ]:

import re
from collections import Counter

def looks_like_citation(s):
    s = s.strip()
    return bool(
        re.search(r"\bv\.?\s", s, re.I)
        or re.search(r"\b(SCC|AIR|SCR|MANU)\b", s)
        or re.search(r"\b\d{4}\b", s)
    )

def looks_like_person(s):
    words = re.findall(r"[A-Za-z][A-Za-z.'’-]*", s)
    return (
        1 <= len(words) <= 6
        and all(w[0].isupper() for w in words)
        and not looks_like_citation(s)
    )

def suspicious_replacement(label, old, new):
    label = label.upper()
    old_citation = looks_like_citation(old)
    new_citation = looks_like_citation(new)

    if label == "PREC":
        return old_citation != new_citation
    if label == "DATE":
        return not bool(re.search(
            r"\d|January|February|March|April|May|June|July|August|"
            r"September|October|November|December",
            new, re.I
        ))
    if label == "CASENO":
        return not bool(re.search(r"\d", new))
    if label in {"APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT"}:
        return not looks_like_person(new)
    if label == "STAT":
        return not bool(re.search(
            r"\b(Act|Code|Rules|Constitution|Regulation|Ordinance)\b",
            new, re.I
        ))
    return False

flagged = [
    x for x in replacement_audit
    if suspicious_replacement(x["label"], x["old"], x["new"])
]

print("Replacements flagged for review:", len(flagged), "/", len(replacement_audit))
print("\nFlagged samples:")
for x in flagged[:60]:
    print(f'{x["label"]:12s} | {x["old"]!r} -> {x["new"]!r}')


In [ ]:

# Corrected token/BIO integrity audit.
# Overlapping chunks are expected; validate each chunk independently.

valid_ids = set(label2id.values())
errors = []
total_tokens = 0
total_entities = 0
ignored_tokens = 0
chunk_count = 0

for doc_id, text in aug_texts.items():
    entities = aug_gold[doc_id]
    chunks, tags = encode_train_doc(text, entities)

    enc = tokenizer.encode(text, add_special_tokens=False)
    ids, offsets = list(enc.ids), list(enc.offsets)

    if len(ids) != len(tags):
        errors.append((doc_id, "token/tag length mismatch"))
        continue

    if any(t != -100 and t not in valid_ids for t in tags):
        errors.append((doc_id, "invalid BIO label ID"))

    # Validate each chunk against its corresponding original token window.
    expected_start = 0
    for chunk in chunks:
        chunk_count += 1
        chunk_ids = chunk["input_ids"]
        chunk_labels = chunk["labels"]

        if len(chunk_ids) != len(chunk_labels):
            errors.append((doc_id, "chunk input/label length mismatch"))
            continue

        if len(chunk_ids) > MAX_LEN:
            errors.append((doc_id, "chunk exceeds MAX_LEN"))

        if not chunk_ids:
            errors.append((doc_id, "empty chunk"))
            continue

        # Locate this chunk's token window in the original sequence.
        # The encoder uses starts at multiples of STRIDE.
        start = expected_start
        end = start + len(chunk_ids)

        if ids[start:end] != chunk_ids:
            errors.append((doc_id, "chunk tokens do not match expected window"))
            continue

        if end > len(ids):
            errors.append((doc_id, "chunk extends past token sequence"))
            continue

        expected_start += STRIDE
        if end == len(ids):
            expected_start = len(ids)

    # Verify the chunks cover the complete token sequence.
    if ids and not chunks:
        errors.append((doc_id, "no chunks returned"))
    elif chunks and len(ids) > MAX_LEN:
        last_start = (len(chunks) - 1) * STRIDE
        if last_start + len(chunks[-1]["input_ids"]) < len(ids):
            errors.append((doc_id, "last chunk does not cover document end"))

    # Every valid entity must overlap at least one token.
    for ent in entities:
        s, e, lab = int(ent[0]), int(ent[1]), str(ent[2])

        if lab not in ENTITY_LABELS or s < 0 or e <= s or e > len(text):
            errors.append((doc_id, f"invalid entity span: {ent[:3]}"))
            continue

        overlap = [
            i for i, (a, b) in enumerate(offsets)
            if b > a and a < e and b > s
        ]
        if not overlap:
            errors.append((doc_id, f"entity has no token overlap: {ent[:3]}"))

        total_entities += 1

    total_tokens += len(ids)
    ignored_tokens += sum(t == -100 for t in tags)

print("Documents checked:", len(aug_texts))
print("Chunks checked:", chunk_count)
print("Tokens checked:", total_tokens)
print("Entities checked:", total_entities)
print("Ignored/conflict tokens:", ignored_tokens)
print("Errors:", len(errors))

for err in errors[:30]:
    print("ERROR:", err)

if not errors:
    print("PASS: token/BIO structural checks passed.")
else:
    print("FAIL: inspect the reported chunk-window errors before training.")


In [ ]:

# Final pre-training audit; does not train or overwrite model artifacts.

assert all(aug_texts[d] == train_texts[d] for d in train_texts)
assert all(d in aug_gold for d in train_texts)

synthetic_ids = [d for d in aug_texts if str(d).endswith("__entity_replacement")]

assert len(aug_texts) == len(train_texts) + len(synthetic_ids)

bad = []
for doc_id in synthetic_ids:
    text = aug_texts[doc_id]
    chunks, tags = encode_train_doc(text, aug_gold[doc_id])

    if not chunks or not tags:
        bad.append((doc_id, "empty encoding"))
        continue

    if len(tags) != len(tokenizer.encode(text, add_special_tokens=False).ids):
        bad.append((doc_id, "token/tag length mismatch"))

print("Original examples preserved:", len(train_texts))
print("Synthetic examples:", len(synthetic_ids))
print("Total examples:", len(aug_texts))
print("Encoding errors:", len(bad))

for item in bad[:20]:
    print("ERROR:", item)

assert not bad, "Resolve encoding errors before training."
print("PASS: final structural pre-training audit.")

In [ ]:

# EXPERIMENT: Entity-replacement augmentation
# Starts from champion weights. Never modifies champion_model or existing predictions.

import copy
import gc
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# ---------- Preconditions ----------
required = [
    "champion_model", "train_texts", "train_gold",
    "val_texts", "val_gold", "aug_texts", "aug_gold",
    "tokenizer", "BIO_LABELS", "label2id", "ENTITY_LABELS",
    "exp_full_pipeline",
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        f"Missing notebook variables: {missing}. Run the preceding setup cells first."
    )

synthetic_ids = [
    d for d in aug_texts
    if str(d).endswith("__entity_replacement")
]
assert len(train_texts) == 35, f"Expected 35 original training docs, got {len(train_texts)}"
assert len(synthetic_ids) > 0, "No synthetic variants found."
assert all(aug_texts[d] == train_texts[d] for d in train_texts)
assert set(train_texts).isdisjoint(set(val_texts))
assert set(val_texts) == set(val_gold)

# Fixed experiment settings taken from the existing notebook.
SEED_AUG_EXP = 78121
AUG_MAX_LEN = 512
AUG_STRIDE = 448
AUG_BATCH_SIZE = 1
AUG_EPOCHS = 1
AUG_ENCODER_LR = 5e-6
AUG_CLASSIFIER_LR = 2e-5

random.seed(SEED_AUG_EXP)
np.random.seed(SEED_AUG_EXP)
torch.manual_seed(SEED_AUG_EXP)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED_AUG_EXP)

# Reconfirm the reference pipeline before training.
champion_model.eval()
_, aug_reference_metrics = exp_full_pipeline(
    champion_model, val_texts, val_gold
)
aug_reference_score = float(aug_reference_metrics["MACRO_F1"])

print(f"Champion hybrid validation macro-F1: {aug_reference_score:.6f}")
print("Expected reference: approximately 0.333307")

if abs(aug_reference_score - 0.333307) > 0.002:
    raise RuntimeError(
        "Reference score mismatch. Stop here and investigate before training."
    )

# ---------- Encode original + synthetic examples ----------
def encode_augmented_document(text, entities):
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids, offsets = list(enc.ids), list(enc.offsets)
    tags = [label2id["O"]] * len(ids)
    conflicts = [False] * len(ids)

    for ent in entities:
        if len(ent) < 3:
            continue
        s, e, lab = int(ent[0]), int(ent[1]), str(ent[2])
        if lab not in ENTITY_LABELS or s < 0 or e <= s or e > len(text):
            raise ValueError(f"Invalid entity annotation: {ent[:3]}")

        tok_ix = [
            i for i, (a, b) in enumerate(offsets)
            if b > a and a < e and b > s
        ]
        if not tok_ix:
            raise ValueError(f"Entity has no token overlap: {ent[:3]}")

        for j, i in enumerate(tok_ix):
            tag_id = label2id[("B-" if j == 0 else "I-") + lab]
            if tags[i] != label2id["O"] and tags[i] != tag_id:
                conflicts[i] = True
            else:
                tags[i] = tag_id

    for i, conflict in enumerate(conflicts):
        if conflict:
            tags[i] = -100

    chunks, seen = [], set()
    for start in range(0, len(ids), AUG_STRIDE):
        end = min(start + AUG_MAX_LEN, len(ids))
        if start >= end:
            break

        chunk_labels = []
        for i in range(start, end):
            if i in seen:
                chunk_labels.append(-100)
            else:
                chunk_labels.append(tags[i])
                seen.add(i)

        chunks.append({
            "input_ids": ids[start:end],
            "labels": chunk_labels,
        })
        if end == len(ids):
            break

    return chunks

aug_train_chunks = []
for doc_id, text in aug_texts.items():
    if doc_id not in aug_gold:
        raise KeyError(f"Missing annotations for {doc_id}")
    aug_train_chunks.extend(
        encode_augmented_document(text, aug_gold[doc_id])
    )

assert aug_train_chunks, "No training chunks were produced."

def collate_augmented(batch):
    max_len = max(len(item["input_ids"]) for item in batch)
    ids = torch.zeros((len(batch), max_len), dtype=torch.long)
    mask = torch.zeros_like(ids)
    labels = torch.full((len(batch), max_len), -100, dtype=torch.long)

    for i, item in enumerate(batch):
        n = len(item["input_ids"])
        ids[i, :n] = torch.tensor(item["input_ids"], dtype=torch.long)
        mask[i, :n] = 1
        labels[i, :n] = torch.tensor(item["labels"], dtype=torch.long)

    return ids, mask, labels

generator = torch.Generator()
generator.manual_seed(SEED_AUG_EXP)

aug_loader = DataLoader(
    aug_train_chunks,
    batch_size=AUG_BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_augmented,
    generator=generator,
)

print("Original documents:", len(train_texts))
print("Synthetic variants:", len(synthetic_ids))
print("Total examples:", len(aug_texts))
print("Training chunks:", len(aug_train_chunks))
print("Validation documents:", len(val_texts))
print("Test data will not be evaluated.")


In [ ]:

# Train a separate experimental model from the untouched champion.
# This cell never assigns to champion_model.

aug_model = copy.deepcopy(champion_model).to(device)
aug_model.train()

optimizer = torch.optim.AdamW(
    [
        {"params": aug_model.encoder.parameters(), "lr": AUG_ENCODER_LR},
        {"params": aug_model.classifier.parameters(), "lr": AUG_CLASSIFIER_LR},
    ],
    weight_decay=0.01,
)
criterion = nn.CrossEntropyLoss(ignore_index=-100)

loss_history = []

for epoch in range(AUG_EPOCHS):
    aug_model.train()
    running_loss = 0.0
    batches = 0

    for ids, mask, labels in tqdm(
        aug_loader,
        desc=f"Entity-replacement experiment: epoch {epoch + 1}/{AUG_EPOCHS}",
    ):
        ids = ids.to(device)
        mask = mask.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = aug_model(ids, mask)

        loss = criterion(
            logits.reshape(-1, len(BIO_LABELS)),
            labels.reshape(-1),
        )

        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite training loss; stopping experiment.")

        loss.backward()
        torch.nn.utils.clip_grad_norm_(aug_model.parameters(), 1.0)
        optimizer.step()

        running_loss += float(loss.item())
        batches += 1

    mean_loss = running_loss / max(batches, 1)
    loss_history.append(mean_loss)
    print(f"Epoch {epoch + 1}: mean training loss = {mean_loss:.6f}")

# Evaluate only on the unchanged validation fold.
aug_model.eval()
aug_val_predictions, aug_val_metrics = exp_full_pipeline(
    aug_model, val_texts, val_gold
)
aug_val_score = float(aug_val_metrics["MACRO_F1"])
aug_delta = aug_val_score - aug_reference_score

print("\n=== VALIDATION RESULTS ===")
print(f"Champion reference macro-F1: {aug_reference_score:.6f}")
print(f"Augmentation macro-F1:       {aug_val_score:.6f}")
print(f"Difference:                  {aug_delta:+.6f}")
print("\nPer-entity exact-span metrics:")
for lab in ENTITY_LABELS:
    m = aug_val_metrics[lab]
    print(
        f'{lab:12s} P={m["precision"]:.4f} '
        f'R={m["recall"]:.4f} F1={m["f1"]:.4f} '
        f'TP={m["TP"]} FP={m["FP"]} FN={m["FN"]}'
    )

# Choose a fresh filename so reruns never overwrite an earlier experiment.
out_dir = Path("/kaggle/working")
base = "experiment_entity_replacement_augmented"
checkpoint_path = out_dir / f"{base}.pt"
version = 2
while checkpoint_path.exists():
    checkpoint_path = out_dir / f"{base}_v{version}.pt"
    version += 1

torch.save(
    {
        "model_state_dict": {
            k: v.detach().cpu()
            for k, v in aug_model.state_dict().items()
        },
        "bio_labels": BIO_LABELS,
        "experiment": "entity_replacement_augmentation",
        "training_documents_including_synthetic": len(aug_texts),
        "original_training_documents": len(train_texts),
        "synthetic_documents": len(synthetic_ids),
        "epochs": AUG_EPOCHS,
        "encoder_lr": AUG_ENCODER_LR,
        "classifier_lr": AUG_CLASSIFIER_LR,
        "training_loss": loss_history,
        "validation_macro_f1": aug_val_score,
        "reference_macro_f1": aug_reference_score,
        "validation_delta": aug_delta,
        "validation_metrics": aug_val_metrics,
    },
    checkpoint_path,
)

metrics_path = checkpoint_path.with_suffix(".metrics.json")
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "experiment": "entity_replacement_augmentation",
            "validation_macro_f1": aug_val_score,
            "reference_macro_f1": aug_reference_score,
            "validation_delta": aug_delta,
            "training_loss": loss_history,
            "validation_metrics": aug_val_metrics,
            "test_evaluated": False,
        },
        f,
        indent=2,
    )

print("\nSaved experimental checkpoint:", checkpoint_path)
print("Saved validation metrics:", metrics_path)
print("Champion checkpoint untouched:", CHAMPION_PATH)
print("Existing prediction JSON files untouched.")
print("No test labels or test-set evaluation were used.")


In [ ]:
import inspect
print(inspect.getsource(exp_full_pipeline))

In [ ]:

import pandas as pd
from pathlib import Path

review_rows = []

for doc_id in train_texts:
    synthetic_id = f"{doc_id}__entity_replacement"
    if synthetic_id not in aug_texts:
        continue

    old_text = train_texts[doc_id]
    new_text = aug_texts[synthetic_id]
    old_spans = train_gold[str(doc_id)] if str(doc_id) in train_gold else train_gold[doc_id]
    new_spans = aug_gold[synthetic_id]

    for old, new in zip(old_spans, new_spans):
        os_, oe, olab = int(old[0]), int(old[1]), str(old[2])
        ns, ne, nlab = int(new[0]), int(new[1]), str(new[2])

        old_surface = old_text[os_:oe]
        new_surface = new_text[ns:ne]

        if old_surface == new_surface:
            continue

        review_rows.append({
            "doc_id": str(doc_id),
            "label": olab,
            "original_surface": old_surface,
            "replacement_surface": new_surface,
            "original_context": old_text[max(0, os_-100):min(len(old_text), oe+100)],
            "synthetic_context": new_text[max(0, ns-100):min(len(new_text), ne+100)],
            "label_unchanged": olab == nlab,
            "surface_changed": old_surface != new_surface,
        })

review_df = pd.DataFrame(review_rows)
review_path = Path("/kaggle/working/entity_replacement_manual_review.csv")
review_df.to_csv(review_path, index=False)

print("Changed mentions:", len(review_df))
print("\nReplacements by label:")
print(review_df.groupby("label").size().sort_values(ascending=False).to_string())
print("\nReview file:", review_path)
display(review_df.head(20))


In [ ]:

# Diagnostic only: no training, no checkpoint writes, no test evaluation.

def normalise_gold_for_metrics(gold):
    return {
        str(doc_id): [
            (int(span[0]), int(span[1]), str(span[2]))
            for span in spans
        ]
        for doc_id, spans in gold.items()
    }

gold_eval = normalise_gold_for_metrics(val_gold)

def to_span_predictions(predictions):
    result = {}
    for doc_id, spans in predictions.items():
        result[str(doc_id)] = [
            (int(s), int(e), str(label))
            for s, e, label, *rest in spans
        ]
    return result

def print_macro_and_per_label(name, predictions):
    metrics = calculate_exact_span_metrics(
        gold_eval, predictions, ENTITY_LABELS
    )
    print(f"\n{name}: macro-F1 = {metrics['MACRO_F1']:.6f}")
    for label in ENTITY_LABELS:
        m = metrics[label]
        print(
            f"{label:12s} F1={m['f1']:.4f} "
            f"P={m['precision']:.4f} R={m['recall']:.4f} "
            f"TP={m['TP']} FP={m['FP']} FN={m['FN']}"
        )
    return metrics

# Raw model predictions before thresholding/rules.
champion_raw = predict_texts(champion_model, val_texts)
augmented_raw = predict_texts(aug_model, val_texts)

champion_raw_spans = to_span_predictions(champion_raw)
augmented_raw_spans = to_span_predictions(augmented_raw)

champion_raw_metrics = print_macro_and_per_label(
    "Champion — raw predictions",
    champion_raw_spans
)
augmented_raw_metrics = print_macro_and_per_label(
    "Augmented — raw predictions",
    augmented_raw_spans
)

# Existing full post-processing pipeline, evaluated consistently.
_, champion_full_metrics = exp_full_pipeline(
    champion_model, val_texts, val_gold
)
_, augmented_full_metrics = exp_full_pipeline(
    aug_model, val_texts, val_gold
)

print("\n=== POST-PROCESSED COMPARISON ===")
print("Champion:", round(champion_full_metrics["MACRO_F1"], 6))
print("Augmented:", round(augmented_full_metrics["MACRO_F1"], 6))


In [ ]:

# Compare model parameters without modifying either model.

import torch

champion_state = champion_model.state_dict()
augmented_state = aug_model.state_dict()

if champion_state.keys() != augmented_state.keys():
    raise RuntimeError("Champion and augmented model state dictionaries differ.")

rows = []
total_diff_sq = 0.0
total_base_sq = 0.0

for name in champion_state:
    a = champion_state[name].detach().float().cpu()
    b = augmented_state[name].detach().float().cpu()

    diff_norm = torch.linalg.vector_norm(b - a).item()
    base_norm = torch.linalg.vector_norm(a).item()

    total_diff_sq += diff_norm ** 2
    total_base_sq += base_norm ** 2

    rows.append({
        "parameter": name,
        "champion_norm": base_norm,
        "change_norm": diff_norm,
        "relative_change": diff_norm / max(base_norm, 1e-12),
    })

weight_report = pd.DataFrame(rows)
weight_report = weight_report.sort_values(
    "relative_change", ascending=False
)

print(
    "Overall relative parameter change:",
    (total_diff_sq ** 0.5) / max(total_base_sq ** 0.5, 1e-12)
)
display(weight_report.head(25))


In [ ]:
# CELL 1 — stable training harness: definitions + champion reference check (trains nothing, writes nothing but a ledger folder)
import gc, torch
need = ["champion_model", "tokenizer", "BIO_LABELS", "label2id", "ENTITY_LABELS", "train_texts", "train_gold", "val_texts",
        "val_gold", "phase1_state", "gpt2_config", "LegalNERModel", "exp_full_pipeline", "predict_texts", "apply_thr",
        "calculate_exact_span_metrics", "VERIFIED_THR", "DIAG_MAX_LEN", "DIAG_STRIDE", "device", "merge_all_original",
        "val_rule_candidates"]
missing = [n for n in need if n not in globals()]
assert not missing, f"missing from the session: {missing} (run the earlier finalv2 cells first; cell 12 defines exp_full_pipeline)"
for n in ["expA", "expB", "results", "aug_model", "optimizer", "aug_loader", "train_loader"]:   # free leftover GPU objects
    globals().pop(n, None)
gc.collect(); torch.cuda.empty_cache()

# ---------- PART A: data utilities (pure python) ----------
import re, copy, random, math, time, json, types
from bisect import bisect_left, bisect_right
from collections import Counter, defaultdict
from pathlib import Path

N_LAB = len(BIO_LABELS); O_ID = label2id["O"]
MODEL_LABELS  = ["APP", "RESP", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"]   # labels the network itself must produce
PERSON_LABELS = {"APP", "RESP", "WIT", "A.COUNSEL", "R.COUNSEL", "JUDGE"}

def tokenize_with_tags(text, entities):
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids, offs = list(enc.ids), list(enc.offsets)
    starts = [a for a, b in offs]; ends = [b for a, b in offs]
    tags = [O_ID] * len(ids); bad = [False] * len(ids)
    for ent in entities:
        s, e, lab = int(ent[0]), int(ent[1]), str(ent[2])
        if lab not in ENTITY_LABELS or s < 0 or e <= s or e > len(text):
            continue
        ix = [i for i in range(bisect_right(ends, s), bisect_left(starts, e)) if ends[i] > starts[i]]
        for j, i in enumerate(ix):
            t = label2id[("B-" if j == 0 else "I-") + lab]
            if tags[i] != O_ID and tags[i] != t: bad[i] = True
            else: tags[i] = t
    return ids, [(-100 if b else t) for t, b in zip(tags, bad)]

def make_chunks(ids, tags, L, stride, echo):
    """Overlapping windows. Every position is supervised in every window it appears in (inference averages
    overlapping windows, so training mirrors it). echo=True: input is the window repeated twice and only the
    second copy is supervised, so each supervised token has seen the whole window (left AND right)."""
    chunks, n = [], len(ids)
    for st in range(0, n, stride):
        en = min(st + L, n)
        if st >= en: break
        cid, lab = ids[st:en], tags[st:en]
        if echo: chunks.append({"input_ids": cid + cid, "labels": [-100] * len(cid) + lab})
        else:    chunks.append({"input_ids": cid, "labels": lab})
        if en == n: break
    return chunks

def build_chunks(texts, gold, L, stride, echo):
    out = []
    for d, text in texts.items():
        ids, tags = tokenize_with_tags(text, gold[d])
        out += make_chunks(ids, tags, L, stride, echo)
    return out

# ---- consistent mention replacement (name-swap) augmentation: pools come from TRAIN gold only ----
ORG_WORDS = re.compile(r"\b(state|union|india|government|govt|corporation|court|department|commission|board|ltd|limited|"
                       r"company|bank|municipal|authority|committee|association|council|society|ministry|police|custodian|"
                       r"collector|officer|director|secretary|commissioner|university|trust|institute|ors|anr|others|another)\b", re.I)
NAME_SHAPE = re.compile(r"[A-Z][A-Za-z\.'’\-]*(?: [A-Z][A-Za-z\.'’\-]*){0,3}")

def is_person_like(s): return bool(NAME_SHAPE.fullmatch(s.strip())) and not ORG_WORDS.search(s)

def group_key(surface, lab):
    if lab in PERSON_LABELS: return "PERSON" if is_person_like(surface) else "ORG:" + lab
    return lab

def build_pools(texts, gold):
    pools = defaultdict(set)
    for d, ents in gold.items():
        for ent in ents:
            s, e, lab = int(ent[0]), int(ent[1]), str(ent[2])
            surf = texts[d][s:e]
            if surf.strip(): pools[group_key(surf, lab)].add(surf)
    return {k: sorted(v) for k, v in pools.items()}

def augment_doc(text, ents, pools, p, rng):
    spans = sorted(((int(x[0]), int(x[1]), str(x[2])) for x in ents), key=lambda x: (x[0], x[1]))
    clash = [any(j != i and spans[j][0] < e and spans[j][1] > s for j in range(len(spans)))
             for i, (s, e, _) in enumerate(spans)]
    choice = {}
    for (s, e, lab), bad in zip(spans, clash):
        if bad: continue
        key = (group_key(text[s:e], lab), text[s:e])
        if key in choice: continue
        pool = [x for x in pools.get(key[0], []) if x != text[s:e]]
        choice[key] = rng.choice(pool) if (pool and rng.random() < p) else None   # same new surface for every mention
    rep = {(s, e): choice[(group_key(text[s:e], lab), text[s:e])]
           for (s, e, lab), bad in zip(spans, clash) if not bad and choice[(group_key(text[s:e], lab), text[s:e])] is not None}
    reps = sorted((s, e, new) for (s, e), new in rep.items())
    pieces, cur = [], 0
    for s, e, new in reps:
        pieces.append(text[cur:s]); pieces.append(new); cur = e
    pieces.append(text[cur:]); new_text = "".join(pieces)
    shift = lambda pos: sum(len(new) - (e - s) for s, e, new in reps if e <= pos)
    new_ents = []
    for (s, e, lab) in spans:
        ns = s + shift(s)
        ne = ns + len(rep[(s, e)]) if (s, e) in rep else e + shift(s)
        want = rep.get((s, e)) or text[s:e]
        assert new_text[ns:ne] == want, "offset bookkeeping error"
        new_ents.append([ns, ne, lab])
    return new_text, new_ents, len(rep), len(spans)

def build_augmented(texts, gold, K, p, seed):
    pools = build_pools(texts, gold); rng = random.Random(seed)
    at, ag, changed, total = {}, {}, 0, 0
    for k in range(K):
        for d, text in texts.items():
            nt, ne, c, t = augment_doc(text, gold[d], pools, p, rng)
            at[f"{d}__mr{k}"], ag[f"{d}__mr{k}"] = nt, ne; changed += c; total += t
    print(f"augmentation: {len(at)} synthetic docs; {changed}/{total} mentions replaced ({changed/max(total,1):.1%}); "
          f"pools: " + ", ".join(f"{k}={len(v)}" for k, v in sorted(pools.items())))
    return at, ag
# ---------- PART B: model, training, evaluation ----------
import gc
import numpy as np, pandas as pd, torch, torch.nn as nn
from transformers import GPT2Model

EXP_DIR = Path("/kaggle/working/exps_v2"); EXP_DIR.mkdir(exist_ok=True, parents=True)
LEDGER = EXP_DIR / "ledger.csv"
DEFAULTS = dict(seed=0, L=512, stride=448, echo=False, aug=None, epochs=4, lr_body=3e-5, lr_head=1e-3, wd=0.01,
                accum=8, warmup=0.1, ema=0.98, dropout=0.0, weights="champion", eval_epochs=None, save=False)
THR_GRID = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]

class EchoWrap(nn.Module):
    """Repeat-the-input trick at inference, so the existing pipeline (windows, Viterbi, thresholds, rules) is unchanged."""
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x, mask=None):
        n = x.shape[1]; xx = torch.cat([x, x], dim=1)
        return self.m(xx, torch.ones_like(xx))[:, n:]

def with_windows(L, stride, fn):
    global DIAG_MAX_LEN, DIAG_STRIDE
    old = (DIAG_MAX_LEN, DIAG_STRIDE); DIAG_MAX_LEN, DIAG_STRIDE = L, stride
    try: return fn()
    finally: DIAG_MAX_LEN, DIAG_STRIDE = old

def hybrid_metrics(raw, thr):
    """Your validated hybrid (thresholds + counsel/JUDGE/WIT rules + CASENO/STAT trim) on cached raw predictions."""
    global predict_texts, VERIFIED_THR
    orig_p, orig_t = predict_texts, VERIFIED_THR
    predict_texts = lambda m, t: raw
    VERIFIED_THR = thr
    try:
        _, met = exp_full_pipeline(types.SimpleNamespace(eval=lambda: None), val_texts, val_gold)
    finally:
        predict_texts, VERIFIED_THR = orig_p, orig_t
    return met

def tune_thresholds(raw):
    thr = dict(VERIFIED_THR)
    for lab in MODEL_LABELS:
        best = (-1.0, thr[lab])
        for t in THR_GRID:
            pred = {d: [(s, e, l) for s, e, l, c in sp if l == lab and c >= t] for d, sp in raw.items()}
            f = calculate_exact_span_metrics(val_gold, pred, [lab])[lab]["f1"]
            if f > best[0] + 1e-9: best = (f, t)
        thr[lab] = best[1]
    return thr

def evaluate(model, L, stride, echo):
    net = EchoWrap(model) if echo else model
    raw = with_windows(L, stride, lambda: predict_texts(net, val_texts))
    plain = {d: [(s, e, l) for s, e, l, c in sp] for d, sp in raw.items()}
    m_raw = calculate_exact_span_metrics(val_gold, plain, ENTITY_LABELS)
    tuned = tune_thresholds(raw)
    m_ch, m_tu = hybrid_metrics(raw, VERIFIED_THR), hybrid_metrics(raw, tuned)
    return {"model8": float(np.mean([m_raw[l]["f1"] for l in MODEL_LABELS])),
            "hybrid_champ_thr": float(m_ch["MACRO_F1"]), "hybrid_tuned": float(m_tu["MACRO_F1"]),
            "raw_f1": {l: m_raw[l]["f1"] for l in MODEL_LABELS},
            "tuned_f1": {l: m_tu[l]["f1"] for l in ENTITY_LABELS}, "thr": tuned}

def new_model(dropout=0.0):
    c = copy.deepcopy(gpt2_config)
    for a in ("resid_pdrop", "embd_pdrop", "attn_pdrop"): setattr(c, a, dropout)
    enc = GPT2Model(c); enc.load_state_dict(phase1_state, strict=True)          # always from the Phase 1 weights
    wrapper = type("LMWrapper", (), {"transformer": enc, "config": c})()
    return LegalNERModel(wrapper, N_LAB).to(device)

def champion_style_weights(chunks):
    cnt = torch.zeros(N_LAB)
    for ch in chunks:
        lab = torch.tensor([x for x in ch["labels"] if x >= 0], dtype=torch.long)
        cnt += torch.bincount(lab, minlength=N_LAB).float()
    w = torch.sqrt(cnt[O_ID] / cnt.clamp_min(1)); w[O_ID] = 1.0
    w = w.clamp(1.0, 5.0)
    return (w / w.mean()).to(device)

@torch.no_grad()
def ema_update(ema, model, d):
    for k, v in model.state_dict().items():
        if v.dtype.is_floating_point: ema[k].mul_(d).add_(v.detach(), alpha=1 - d)
        else: ema[k].copy_(v)

def eval_with_weights(model, weights, fn):
    backup = {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(weights)
    try: return fn()
    finally: model.load_state_dict(backup)

def run_cfg(name, **cfg):
    c = dict(DEFAULTS); c.update(cfg); t0 = time.time()
    torch.manual_seed(c["seed"]); np.random.seed(c["seed"]); random.seed(c["seed"])
    gc.collect(); torch.cuda.empty_cache()
    texts = {str(k): v for k, v in train_texts.items()}; gold = {str(k): v for k, v in train_gold.items()}
    if c["aug"]:
        at, ag = build_augmented(texts, gold, K=c["aug"]["K"], p=c["aug"]["p"], seed=c["seed"] + 1000)
        texts.update(at); gold.update(ag)
    chunks = build_chunks(texts, gold, c["L"], c["stride"], c["echo"])
    if c["echo"]: assert 2 * c["L"] <= gpt2_config.n_positions, "echo input exceeds the context length"
    print(f"[{name}] docs={len(texts)} chunks={len(chunks)} L={c['L']} stride={c['stride']} echo={c['echo']} "
          f"epochs={c['epochs']} seed={c['seed']}", flush=True)
    crit = nn.CrossEntropyLoss(weight=champion_style_weights(chunks) if c["weights"] == "champion" else None, ignore_index=-100)
    model = new_model(c["dropout"])
    body = [p for n, p in model.named_parameters() if n.startswith("encoder.")]
    head = [p for n, p in model.named_parameters() if not n.startswith("encoder.")]
    opt = torch.optim.AdamW([{"params": body, "lr": c["lr_body"]}, {"params": head, "lr": c["lr_head"]}], weight_decay=c["wd"])
    spe = math.ceil(len(chunks) / c["accum"]); total = spe * c["epochs"]; warm = max(1, int(c["warmup"] * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / warm, max(0.0, (total - s) / max(1, total - warm))))
    ema = {k: v.detach().clone() for k, v in model.state_dict().items()}
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    g = torch.Generator().manual_seed(c["seed"])
    eval_epochs = sorted(set((c["eval_epochs"] or []) + [c["epochs"]]))
    result = None
    for epoch in range(1, c["epochs"] + 1):
        model.train(); opt.zero_grad(set_to_none=True); run_loss = 0.0
        order = torch.randperm(len(chunks), generator=g).tolist()
        for step, ci in enumerate(order, 1):
            x = torch.tensor([chunks[ci]["input_ids"]], dtype=torch.long, device=device)
            y = torch.tensor([chunks[ci]["labels"]], dtype=torch.long, device=device)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                logits = model(x, torch.ones_like(x))
            loss = crit(logits.float().reshape(-1, N_LAB), y.reshape(-1))
            if not torch.isfinite(loss): raise RuntimeError("non-finite loss")
            scaler.scale(loss / c["accum"]).backward(); run_loss += loss.item()
            if step % c["accum"] == 0 or step == len(order):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
                ema_update(ema, model, c["ema"])
        msg = f"[{name}] epoch {epoch}/{c['epochs']} loss {run_loss/len(order):.4f} ({(time.time()-t0)/60:.1f} min)"
        if epoch in eval_epochs:
            result = eval_with_weights(model, ema, lambda: evaluate(model, c["L"], c["stride"], c["echo"]))
            model.train()
            msg += (f" | model8={result['model8']:.4f}  hybrid(champ thr)={result['hybrid_champ_thr']:.4f}"
                    f"  hybrid(tuned thr)={result['hybrid_tuned']:.4f}")
        print(msg, flush=True)
    if c["save"]:
        torch.save({"model_state_dict": {k: v.cpu() for k, v in ema.items()}, "cfg": c}, EXP_DIR / f"{name}.pt")
    row = {"name": name, "seed": c["seed"], "minutes": round((time.time() - t0) / 60, 1), "cfg": json.dumps(c),
           "model8": result["model8"], "hybrid_champ_thr": result["hybrid_champ_thr"], "hybrid_tuned": result["hybrid_tuned"],
           **{f"raw_{l}": v for l, v in result["raw_f1"].items()}, **{f"tuned_{l}": v for l, v in result["tuned_f1"].items()}}
    pd.DataFrame([row]).to_csv(LEDGER, mode="a", header=not LEDGER.exists(), index=False)
    del model, opt, ema, scaler; gc.collect(); torch.cuda.empty_cache()
    return result

def show_ledger():
    df = pd.read_csv(LEDGER)
    cols = ["name", "seed", "minutes", "model8", "hybrid_champ_thr", "hybrid_tuned"] + [f"raw_{l}" for l in MODEL_LABELS]
    print(df[cols].round(4).to_string(index=False))

# ---------- reference: the champion under THIS harness ----------
champion_model.eval()
ref = evaluate(champion_model, 512, 448, False)
print(f"CHAMPION (fold 2, 512/448 windows): model8={ref['model8']:.4f} | hybrid(champion thr)={ref['hybrid_champ_thr']:.4f} "
      f"| hybrid(thresholds re-tuned on fold 2)={ref['hybrid_tuned']:.4f}")
print("raw F1 per model label:", {k: round(v, 3) for k, v in ref["raw_f1"].items()})
assert abs(ref["hybrid_champ_thr"] - 0.333307) < 0.002, "harness does not reproduce 0.333307 - stop and paste this output"
print("harness reproduces the reference; nothing was trained.")

In [ ]:
# CELL 2 — controls: fresh Phase 1 start, stable recipe, two seeds (~11 min). Gives the unselected baseline and the noise level.
run_cfg("B512_s0", seed=0)
run_cfg("B512_s1", seed=1)
show_ledger()

In [ ]:
# CELL 3 — window-length control vs echo (~18 min)
run_cfg("B256_s0",    seed=0, L=256, stride=192)
run_cfg("ECHO256_s0", seed=0, L=256, stride=192, echo=True)
show_ledger()

In [ ]:
# CELL 4 — real name-swap augmentation (~12 min)
run_cfg("B512_MR_s0", seed=0, aug={"K": 2, "p": 0.5})
show_ledger()

In [ ]:

# NEXT DIAGNOSTIC: Correct log-probability decoding control.
# No training, no threshold search, no checkpoint or prediction writes.

import torch
import pandas as pd

required = [
    "champion_model", "val_texts", "val_gold",
    "diag_doc_probs", "bio_viterbi_decode",
    "decode_spans", "calculate_exact_span_metrics",
    "ENTITY_LABELS", "BIO_LABELS",
]
missing = [x for x in required if x not in globals()]
assert not missing, f"Missing notebook variables: {missing}"

label_to_idx = {label: i for i, label in enumerate(BIO_LABELS)}
control_predictions = {}
doc_rows = []

champion_model.eval()

for doc_id, text in val_texts.items():
    offsets, probs = diag_doc_probs(champion_model, text)

    if len(offsets) == 0:
        control_predictions[doc_id] = []
        continue

    # Match predict_texts(): Viterbi receives log probabilities.
    emissions = torch.log(probs.clamp_min(1e-12))
    path = bio_viterbi_decode(emissions)

    spans = [
        (int(s), int(e), str(lab))
        for s, e, lab in decode_spans(text, offsets, path)
    ]
    control_predictions[doc_id] = spans

    for target in ("RESP", "WIT"):
        gold_spans = {
            (int(ent[0]), int(ent[1]))
            for ent in val_gold[doc_id]
            if str(ent[2]) == target
        }
        pred_spans = {
            (s, e)
            for s, e, lab in spans
            if lab == target
        }

        doc_rows.append({
            "doc_id": str(doc_id),
            "label": target,
            "gold": len(gold_spans),
            "pred": len(pred_spans),
            "exact_TP": len(gold_spans & pred_spans),
        })

control_metrics = calculate_exact_span_metrics(
    val_gold, control_predictions, ENTITY_LABELS
)

print(
    "Correct log-probability control macro-F1:",
    round(control_metrics["MACRO_F1"], 6)
)

for target in ("RESP", "WIT"):
    m = control_metrics[target]
    print(
        f"{target}: F1={m['f1']:.4f}, P={m['precision']:.4f}, "
        f"R={m['recall']:.4f}, gold={m['gold']}, "
        f"pred={m['pred']}, TP={m['TP']}"
    )

doc_report = pd.DataFrame(doc_rows)
print("\nPer-document RESP/WIT breakdown:")
display(
    doc_report.sort_values(
        ["label", "exact_TP", "gold"],
        ascending=[True, False, False]
    ).reset_index(drop=True)
)


In [ ]:

import os, json, hashlib, shutil
from pathlib import Path
from datetime import datetime
import torch

ART_DIR = Path("/kaggle/working/cs781_ensemble_archive")
ART_DIR.mkdir(parents=True, exist_ok=True)

def sha256_file(path, block=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(block)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def unique_path(path):
    path = Path(path)
    if not path.exists():
        return path
    i = 2
    while path.with_name(f"{path.stem}_v{i}{path.suffix}").exists():
        i += 1
    return path.with_name(f"{path.stem}_v{i}{path.suffix}")

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
manifest = {
    "created_at": stamp,
    "live_models": [],
    "checkpoint_files": [],
    "notes": [
        "Existing /kaggle/input checkpoints are referenced, not overwritten.",
        "A metric record does not imply model weights were saved.",
    ],
}

# Inventory saved checkpoint files.
for root in (Path("/kaggle/input"), Path("/kaggle/working")):
    if not root.exists():
        continue
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in {".pt", ".pth", ".ckpt"}:
            try:
                manifest["checkpoint_files"].append({
                    "path": str(p),
                    "size_bytes": p.stat().st_size,
                    "sha256": sha256_file(p),
                    "location": "input" if str(p).startswith("/kaggle/input/") else "working",
                })
            except (PermissionError, OSError):
                pass

# Save in-memory models that exist and have not been explicitly archived.
# This includes only actual nn.Module objects, not training metric rows.
candidate_names = [
    "champion_model",
    "aug_model",
    "targeted_model",
    "expA",
    "expB",
    "model",
]

seen_fingerprints = {}
for name in candidate_names:
    obj = globals().get(name)
    if not isinstance(obj, torch.nn.Module):
        continue

    state = {
        k: v.detach().cpu().clone()
        for k, v in obj.state_dict().items()
    }

    h = hashlib.sha256()
    for key in sorted(state):
        h.update(key.encode())
        h.update(state[key].contiguous().numpy().tobytes())
    fingerprint = h.hexdigest()

    entry = {
        "variable": name,
        "fingerprint": fingerprint,
        "class": type(obj).__name__,
        "state_keys": len(state),
    }

    # If another live variable has identical weights, record the alias
    # rather than writing a duplicate large checkpoint.
    if fingerprint in seen_fingerprints:
        entry["alias_of"] = seen_fingerprints[fingerprint]
        manifest["live_models"].append(entry)
        print(f"{name}: identical weights to {seen_fingerprints[fingerprint]}; no duplicate saved")
        continue

    seen_fingerprints[fingerprint] = name
    destination = unique_path(ART_DIR / f"{name}_{stamp}.pt")

    payload = {
        "model_state_dict": state,
        "model_variable": name,
        "model_class": type(obj).__name__,
        "state_fingerprint_sha256": fingerprint,
        "bio_labels": globals().get("BIO_LABELS"),
        "entity_labels": globals().get("ENTITY_LABELS"),
        "saved_at": stamp,
    }

    # Include config if available, without serializing the model object.
    if "gpt2_config" in globals():
        try:
            payload["gpt2_config"] = gpt2_config.to_dict()
        except Exception:
            pass

    torch.save(payload, destination)
    entry["saved_checkpoint"] = str(destination)
    entry["size_bytes"] = destination.stat().st_size
    manifest["live_models"].append(entry)
    print(f"{name}: saved -> {destination}")

# Cache champion validation predictions, if the current inference helpers exist.
if all(k in globals() for k in ["champion_model", "predict_texts", "val_texts"]):
    champion_model.eval()
    raw = predict_texts(champion_model, val_texts)
    cache = {
        str(doc_id): [
            [int(s), int(e), str(lab), float(conf)]
            for s, e, lab, conf in spans
        ]
        for doc_id, spans in raw.items()
    }
    pred_path = unique_path(ART_DIR / f"champion_val_raw_predictions_{stamp}.json")
    with open(pred_path, "w", encoding="utf-8") as f:
        json.dump(cache, f)
    manifest["champion_validation_predictions"] = str(pred_path)
    print("Champion validation predictions saved:", pred_path)

manifest_path = unique_path(ART_DIR / f"checkpoint_manifest_{stamp}.json")
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("\n=== ARCHIVE SUMMARY ===")
print("Saved manifest:", manifest_path)
print("Checkpoint files inventoried:", len(manifest["checkpoint_files"]))
print("Live model entries:", len(manifest["live_models"]))
print("Archive directory:", ART_DIR)


In [ ]:

# CS781: Collect and checkpoint useful NER models for later ensemble experiments.
# - Preserves champion and all existing artifacts.
# - Recovers B256 weights by repeating the already-tested recipe if needed.
# - Archives distinct compatible NER checkpoints and deduplicates exact model states.
# - Does NOT use fold 3 or change the champion.

import os, json, hashlib, shutil, gc
from pathlib import Path
from datetime import datetime
import torch

ARCHIVE = Path("/kaggle/working/cs781_ensemble_archive")
ARCHIVE.mkdir(parents=True, exist_ok=True)
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

def state_fingerprint(state):
    h = hashlib.sha256()
    for key in sorted(state):
        value = state[key]
        if not torch.is_tensor(value):
            continue
        value = value.detach().cpu().contiguous()
        h.update(key.encode())
        h.update(str(value.dtype).encode())
        h.update(str(tuple(value.shape)).encode())
        h.update(value.view(torch.uint8).numpy().tobytes())
    return h.hexdigest()

def safe_load_checkpoint(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

# ------------------------------------------------------------
# 1. Save the strongest fresh-training candidate, B256.
# Its original run used save=False, so the trained weights were
# deleted. Rerun the same recorded configuration only if absent.
# ------------------------------------------------------------
b256_paths = list(Path("/kaggle/working").rglob("B256_s0_saved.pt"))
b256_paths += list(Path("/kaggle/working").rglob("B256_s0_checkpointed_*.pt"))

if not b256_paths:
    if "run_cfg" not in globals():
        print("WARNING: run_cfg is unavailable. B256 cannot be recovered in this cell.")
        print("Run the experiment-harness definition cell, then rerun this cell.")
    else:
        b256_name = f"B256_s0_checkpointed_{STAMP}"
        print("B256 weights were not saved; rerunning the same recorded B256 recipe.")
        b256_result = run_cfg(
            b256_name,
            seed=0,
            L=256,
            stride=192,
            echo=False,
            aug=None,
            save=True,
        )
        print("B256 rerun metrics:", b256_result)

# ------------------------------------------------------------
# 2. Find plausible NER checkpoint files, excluding large
# Phase 1/Phase 4 language-model-only checkpoints.
# ------------------------------------------------------------
roots = [Path("/kaggle/input"), Path("/kaggle/working")]
candidates = set()

known_names = {
    "champion_recovery.pt",
    "champion_best_ner_model.pt",
    "best_ner_model.pt",
    "targeted_pilot_8_steps.pt",
    "experiment_entity_replacement_augmented.pt",
}

for root in roots:
    if not root.exists():
        continue
    for path in root.rglob("*.pt"):
        try:
            size_mib = path.stat().st_size / (1024 ** 2)
        except OSError:
            continue

        # Phase 1/Phase 4 LM checkpoints are roughly 1.3 GiB
        # and do not contain the trained NER classifier head.
        if size_mib > 700:
            continue

        if (
            path.name in known_names
            or "B256_s0_checkpointed_" in path.name
            or path.name == "B256_s0_saved.pt"
        ):
            candidates.add(path)

# Prioritize the established champion as the canonical reference.
def candidate_priority(path):
    name = path.name.lower()
    if name == "champion_recovery.pt":
        return (0, str(path))
    if name == "champion_best_ner_model.pt":
        return (1, str(path))
    if name == "best_ner_model.pt":
        return (2, str(path))
    if "targeted_pilot" in name:
        return (3, str(path))
    if "b256" in name:
        return (4, str(path))
    if "entity_replacement" in name:
        return (5, str(path))
    return (9, str(path))

candidates = sorted(candidates, key=candidate_priority)

# ------------------------------------------------------------
# 3. Keep only compatible trained NER checkpoints.
# Current architecture uses a 25-label BIO classifier over 768
# hidden dimensions. Identical states are saved only once.
# ------------------------------------------------------------
manifest = {
    "created_at": STAMP,
    "archive_directory": str(ARCHIVE),
    "models": [],
    "skipped": [],
    "notes": [
        "Validation metrics are development results, not test scores.",
        "Phase 1/Phase 4 LM-only checkpoints are not NER ensemble members.",
        "A checkpoint with lower validation score is retained only as a research candidate.",
    ],
}

seen = {}
for source in candidates:
    try:
        payload = safe_load_checkpoint(source)
    except Exception as exc:
        manifest["skipped"].append({
            "source": str(source),
            "reason": f"load failed: {type(exc).__name__}: {exc}",
        })
        continue

    if not isinstance(payload, dict):
        manifest["skipped"].append({
            "source": str(source),
            "reason": "checkpoint is not a dictionary",
        })
        continue

    state = payload.get("model_state_dict")
    if not isinstance(state, dict):
        manifest["skipped"].append({
            "source": str(source),
            "reason": "no model_state_dict",
        })
        continue

    classifier_weight = next(
        (
            v for k, v in state.items()
            if k.endswith("classifier.weight") and torch.is_tensor(v)
        ),
        None,
    )

    if (
        classifier_weight is None
        or tuple(classifier_weight.shape) != (25, 768)
    ):
        manifest["skipped"].append({
            "source": str(source),
            "reason": "incompatible or missing 25 x 768 NER classifier",
        })
        del payload
        continue

    fingerprint = state_fingerprint(state)

    if fingerprint in seen:
        manifest["models"].append({
            "name": source.stem,
            "source": str(source),
            "fingerprint": fingerprint,
            "duplicate_of": seen[fingerprint],
            "archived_file": None,
        })
        print(f"Duplicate weights: {source.name} -> {seen[fingerprint]}")
        del payload
        continue

    # Assign descriptive, non-overwriting archive filenames.
    if source.name in {"champion_recovery.pt", "champion_best_ner_model.pt", "best_ner_model.pt"}:
        archive_name = "champion_ner.pt"
        role = "champion"
    elif "targeted_pilot" in source.name.lower():
        archive_name = "targeted_pilot_8_steps.pt"
        role = "experimental_candidate_unverified"
    elif "b256" in source.name.lower():
        archive_name = f"B256_candidate_{STAMP}.pt"
        role = "fresh_training_candidate"
    elif "entity_replacement" in source.name.lower():
        archive_name = "entity_replacement_negative_result.pt"
        role = "negative_result_research_artifact"
    else:
        archive_name = f"{source.stem}_{STAMP}.pt"
        role = "unranked_ner_candidate"

    destination = ARCHIVE / archive_name
    if destination.exists():
        destination = ARCHIVE / f"{Path(archive_name).stem}_{STAMP}.pt"

    # Preserve original checkpoint fields and add provenance.
    archived_payload = dict(payload)
    archived_payload["archive_metadata"] = {
        "source_path": str(source),
        "role": role,
        "state_fingerprint_sha256": fingerprint,
        "archived_at": STAMP,
    }
    if "bio_labels" not in archived_payload and "BIO_LABELS" in globals():
        archived_payload["bio_labels"] = list(BIO_LABELS)
    if "entity_labels" not in archived_payload and "ENTITY_LABELS" in globals():
        archived_payload["entity_labels"] = list(ENTITY_LABELS)

    torch.save(archived_payload, destination)
    seen[fingerprint] = destination.name

    entry = {
        "name": source.stem,
        "role": role,
        "source": str(source),
        "archived_file": str(destination),
        "size_bytes": destination.stat().st_size,
        "fingerprint": fingerprint,
        "cfg": payload.get("cfg"),
        "epoch": payload.get("epoch"),
        "optimizer_steps": payload.get("optimizer_steps"),
        "validation_macro_f1": payload.get("validation_macro_f1"),
    }
    manifest["models"].append(entry)
    print(f"Archived {role}: {source.name} -> {destination.name}")

    del payload, archived_payload, state
    gc.collect()

# ------------------------------------------------------------
# 4. Save the reproducibility metadata for the current champion
# post-processing pipeline, without saving or changing test output.
# ------------------------------------------------------------
postproc = {}
for name in [
    "VERIFIED_THR", "VERIFIED_CHAMPION_THR", "TRIM_CHARS",
    "WIT_EXCLUSIONS", "keep", "chosen",
]:
    if name in globals():
        try:
            value = globals()[name]
            if isinstance(value, set):
                value = sorted(value)
            if isinstance(value, dict):
                value = {
                    str(k): sorted(v) if isinstance(v, set) else v
                    for k, v in value.items()
                }
            json.dumps(value)
            postproc[name] = value
        except Exception:
            postproc[name] = repr(globals()[name])

postproc_path = ARCHIVE / f"champion_postprocessing_{STAMP}.json"
with open(postproc_path, "w", encoding="utf-8") as f:
    json.dump(postproc, f, indent=2, default=str)

manifest["postprocessing_config"] = str(postproc_path)

manifest_path = ARCHIVE / f"model_collection_manifest_{STAMP}.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)

print("\n========== MODEL COLLECTION ==========")
print("Unique NER model states archived:", len(seen))
print("Manifest:", manifest_path)
print("Post-processing metadata:", postproc_path)
print("\nArchived models:")
for item in manifest["models"]:
    print(
        f"- {item['name']} | {item.get('role', 'duplicate')} | "
        f"{item.get('archived_file') or 'duplicate of ' + item.get('duplicate_of', '')}"
    )

print("\nSkipped files:", len(manifest["skipped"]))
for item in manifest["skipped"]:
    print("-", Path(item["source"]).name, ":", item["reason"])

print("\nChampion checkpoint and existing prediction JSON files were not overwritten.")


In [ ]:

# ============================================================
# CS781 — Ensemble readiness audit
# Read-only: checkpoint compatibility + prediction function audit
# ============================================================

import os
import json
import hashlib
import inspect
import torch

ARCHIVE = "/kaggle/working/cs781_ensemble_archive"

CHECKPOINTS = {
    "champion": os.path.join(ARCHIVE, "champion_ner.pt"),
    "targeted_pilot": os.path.join(ARCHIVE, "targeted_pilot_8_steps.pt"),
}

# Add the actual B256 filename if it exists.
for name in os.listdir(ARCHIVE) if os.path.isdir(ARCHIVE) else []:
    if name.startswith("B256_candidate_") and name.endswith(".pt"):
        CHECKPOINTS["B256"] = os.path.join(ARCHIVE, name)
        break

# ------------------------------------------------------------
# 1. Check current notebook functions without executing them
# ------------------------------------------------------------
print("=" * 70)
print("1. NOTEBOOK FUNCTION AVAILABILITY")
print("=" * 70)

FUNCTION_NAMES = [
    "predict_texts",
    "apply_thr",
    "calculate_exact_span_metrics",
    "merge_all_original",
    "run_cfg",
]

for name in FUNCTION_NAMES:
    obj = globals().get(name)
    if callable(obj):
        try:
            signature = str(inspect.signature(obj))
        except Exception:
            signature = "(signature unavailable)"
        print(f"[FOUND] {name}{signature}")
    else:
        print(f"[MISSING] {name}")

# ------------------------------------------------------------
# 2. Check model objects currently in memory
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("2. IN-MEMORY MODELS")
print("=" * 70)

for name in ["champion_model", "model", "aug_model", "expA", "expB"]:
    obj = globals().get(name)
    if obj is not None and hasattr(obj, "state_dict"):
        print(f"[FOUND] {name}: {type(obj).__name__}")
    else:
        print(f"[NOT FOUND] {name}")

# ------------------------------------------------------------
# 3. Inspect archived checkpoint metadata and tensor shapes
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("3. CHECKPOINT COMPATIBILITY")
print("=" * 70)

def load_checkpoint_safely(path):
    # These are your own archived PyTorch checkpoints.
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        return torch.load(path, map_location="cpu")

def get_state_dict(payload):
    if isinstance(payload, dict):
        for key in ("model_state_dict", "state_dict"):
            if isinstance(payload.get(key), dict):
                return payload[key]
        # Some checkpoints may be a raw state dict.
        if payload and all(torch.is_tensor(v) for v in payload.values()):
            return payload
    return None

def fingerprint_state(state):
    h = hashlib.sha256()
    for key in sorted(state):
        value = state[key]
        if torch.is_tensor(value):
            tensor = value.detach().cpu().contiguous()
            h.update(key.encode("utf-8"))
            h.update(str(tuple(tensor.shape)).encode("utf-8"))
            h.update(tensor.numpy().tobytes())
    return h.hexdigest()[:16]

checkpoint_info = {}

for name, path in CHECKPOINTS.items():
    print(f"\n--- {name} ---")
    if not os.path.isfile(path):
        print("[MISSING FILE]", path)
        continue

    try:
        payload = load_checkpoint_safely(path)
        state = get_state_dict(payload)

        if state is None:
            print("[INCOMPATIBLE] No recognizable state dict")
            print("Top-level keys:", list(payload.keys())[:25]
                  if isinstance(payload, dict) else type(payload))
            continue

        shapes = {
            k: tuple(v.shape)
            for k, v in state.items()
            if torch.is_tensor(v)
        }

        classifier_keys = [
            (k, shape) for k, shape in shapes.items()
            if k.endswith("classifier.weight")
        ]

        info = {
            "path": path,
            "fingerprint": fingerprint_state(state),
            "tensor_count": len(shapes),
            "classifier_keys": classifier_keys,
            "bio_labels": payload.get("bio_labels")
                if isinstance(payload, dict) else None,
            "label_count": len(payload["bio_labels"])
                if isinstance(payload, dict)
                and isinstance(payload.get("bio_labels"), (list, tuple))
                else None,
            "state_keys_sample": list(shapes.keys())[:12],
            "state_shapes": shapes,
        }
        checkpoint_info[name] = info

        print("Path:", path)
        print("Fingerprint:", info["fingerprint"])
        print("Tensor count:", info["tensor_count"])
        print("Classifier tensors:", classifier_keys or "Not found")
        print("BIO label count:", info["label_count"])
        print("BIO labels:", info["bio_labels"])
        print("Sample parameter names/shapes:")
        for key, shape in list(shapes.items())[:12]:
            print(" ", key, shape)

    except Exception as e:
        print("[LOAD ERROR]", repr(e))

# ------------------------------------------------------------
# 4. Compare checkpoint structures and labels
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("4. PAIRWISE STRUCTURAL COMPATIBILITY")
print("=" * 70)

names = list(checkpoint_info)
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = names[i], names[j]
        ia, ib = checkpoint_info[a], checkpoint_info[b]

        same_keys = set(ia["state_shapes"]) == set(ib["state_shapes"])
        same_shapes = (
            same_keys
            and all(
                ia["state_shapes"][k] == ib["state_shapes"][k]
                for k in ia["state_shapes"]
            )
        )

        la, lb = ia["bio_labels"], ib["bio_labels"]
        labels_match = (
            la == lb if la is not None and lb is not None
            else None
        )

        print(
            f"{a} vs {b}: "
            f"same_weights={ia['fingerprint'] == ib['fingerprint']}, "
            f"same_parameter_shapes={same_shapes}, "
            f"labels_match={labels_match}"
        )

# ------------------------------------------------------------
# 5. Check tokenizer and model-building context
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("5. TOKENIZATION / MODEL CONTEXT")
print("=" * 70)

for name in [
    "tokenizer", "tok", "legal_tokenizer",
    "model_config", "config", "bio_labels",
    "LABELS", "id2label", "label2id",
    "train_texts", "val_texts", "val_gold",
    "test_texts",
]:
    obj = globals().get(name)
    if obj is None:
        print(f"[NOT FOUND] {name}")
    else:
        if isinstance(obj, dict):
            print(f"[FOUND] {name}: dict with {len(obj)} entries")
        elif isinstance(obj, (list, tuple)):
            print(f"[FOUND] {name}: {type(obj).__name__}, length={len(obj)}")
        else:
            print(f"[FOUND] {name}: {type(obj).__name__}")

# Print prediction function source to determine whether it exposes
# token-level probabilities or only decoded spans.
print("\n" + "=" * 70)
print("6. PREDICTOR SOURCE AUDIT")
print("=" * 70)

predictor = globals().get("predict_texts")
if callable(predictor):
    try:
        source = inspect.getsource(predictor)
        print(source[:12000])
    except Exception as e:
        print("Could not retrieve source:", repr(e))
else:
    print("predict_texts is unavailable in this kernel.")

# ------------------------------------------------------------
# 7. Save a small audit report (does not modify model files)
# ------------------------------------------------------------
report = {
    "checkpoint_paths": {
        name: info["path"] for name, info in checkpoint_info.items()
    },
    "checkpoint_fingerprints": {
        name: info["fingerprint"] for name, info in checkpoint_info.items()
    },
    "checkpoint_tensor_counts": {
        name: info["tensor_count"] for name, info in checkpoint_info.items()
    },
    "checkpoint_label_counts": {
        name: info["label_count"] for name, info in checkpoint_info.items()
    },
    "functions_found": {
        name: callable(globals().get(name)) for name in FUNCTION_NAMES
    },
}

report_path = os.path.join(ARCHIVE, "ensemble_readiness_audit.json")
os.makedirs(ARCHIVE, exist_ok=True)
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, default=str)

print("\n" + "=" * 70)
print("AUDIT COMPLETE")
print("Report:", report_path)
print("No training was run.")
print("No checkpoint or prediction file was modified.")


In [ ]:

# ============================================================
# CS781 — Controlled probability-ensemble validation
# Models: Champion, B256, targeted pilot, equal-probability mixes
# Validation only; fixed champion thresholds; no training.
# ============================================================

import os
import json
import copy
import torch
import numpy as np
from tqdm.auto import tqdm
from bisect import bisect_left, bisect_right

ARCHIVE = "/kaggle/working/cs781_ensemble_archive"
os.makedirs(ARCHIVE, exist_ok=True)

MODEL_PATHS = {
    "champion": os.path.join(ARCHIVE, "champion_ner.pt"),
    "B256": os.path.join(
        ARCHIVE, "B256_candidate_20261011_170655.pt"
    ),
    "targeted_pilot": os.path.join(
        ARCHIVE, "targeted_pilot_8_steps.pt"
    ),
}

# Use the previously verified champion thresholds; do not tune them here.
if "VERIFIED_CHAMPION_THR" in globals():
    FIXED_THRESHOLDS = dict(VERIFIED_CHAMPION_THR)
elif "VERIFIED_THR" in globals():
    FIXED_THRESHOLDS = dict(VERIFIED_THR)
else:
    raise RuntimeError(
        "Verified champion thresholds are missing. "
        "Refusing to invent thresholds or tune them in this cell."
    )

required = [
    "champion_model", "diag_doc_probs", "bio_viterbi_decode",
    "decode_spans", "calculate_exact_span_metrics",
    "val_texts", "val_gold", "id2label"
]
missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(f"Missing notebook variables/functions: {missing}")

# ------------------------------------------------------------
# 1. Load compatible checkpoint copies
# ------------------------------------------------------------
device = next(champion_model.parameters()).device
print("Inference device:", device)
print("Validation documents:", len(val_texts))
print("Fixed threshold source: verified champion thresholds")

def safe_load(path):
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Checkpoint missing: {path}")
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        return torch.load(path, map_location="cpu")

def get_state(payload):
    if isinstance(payload, dict):
        if isinstance(payload.get("model_state_dict"), dict):
            return payload["model_state_dict"]
        if isinstance(payload.get("state_dict"), dict):
            return payload["state_dict"]
    raise ValueError("Checkpoint has no model_state_dict/state_dict")

def load_model_copy(path):
    payload = safe_load(path)
    state = get_state(payload)

    # Verify the BIO label ordering before loading any candidate.
    saved_labels = payload.get("bio_labels") if isinstance(payload, dict) else None
    if saved_labels is not None:
        current_labels = [id2label[i] for i in sorted(id2label)]
        if list(saved_labels) != list(current_labels):
            raise ValueError(f"BIO label ordering mismatch: {path}")

    candidate = copy.deepcopy(champion_model)
    candidate.load_state_dict(state, strict=True)
    candidate.to(device)
    candidate.eval()
    return candidate

models = {}
for name, path in MODEL_PATHS.items():
    models[name] = load_model_copy(path)
    print(f"[LOADED] {name}")

# ------------------------------------------------------------
# 2. Get token probabilities from every model
# ------------------------------------------------------------
# The existing diagnostic function returns (character_offsets, probabilities).
# Cache on CPU so the GPU only holds the models and one document at a time.
prob_cache = {name: {} for name in models}
offset_cache = {}

for doc_id, text in tqdm(val_texts.items(), desc="Computing validation probabilities"):
    for name, net in models.items():
        with torch.inference_mode():
            offsets, probs = diag_doc_probs(net, text)

        offsets = [(int(s), int(e)) for s, e in offsets]
        if torch.is_tensor(probs):
            probs = probs.detach().float().cpu().numpy()
        else:
            probs = np.asarray(probs, dtype=np.float32)

        if probs.ndim != 2 or probs.shape[0] != len(offsets):
            raise ValueError(
                f"{name}/{doc_id}: offsets/probability shape mismatch: "
                f"{len(offsets)} offsets, probs={probs.shape}"
            )

        if probs.shape[1] != len(id2label):
            raise ValueError(
                f"{name}/{doc_id}: expected {len(id2label)} labels, "
                f"got {probs.shape[1]}"
            )

        if doc_id not in offset_cache:
            offset_cache[doc_id] = offsets
        elif offset_cache[doc_id] != offsets:
            raise ValueError(
                f"Token offsets differ for {name}/{doc_id}; "
                "probability averaging would be invalid."
            )

        prob_cache[name][doc_id] = probs.astype(np.float32, copy=False)

print("Probability extraction complete. Token offsets aligned.")

# Release GPU copies; all required validation probabilities are cached on CPU.
for net in models.values():
    net.to("cpu")
del models
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ------------------------------------------------------------
# 3. Convert probabilities to spans using existing BIO decoding
# ------------------------------------------------------------
def probabilities_to_raw_spans(text, offsets, probs):
    if len(offsets) == 0:
        return []

    probs_t = torch.as_tensor(probs, dtype=torch.float32)
    pred_ids = torch.tensor(
        bio_viterbi_decode(torch.log(probs_t.clamp_min(1e-12))),
        dtype=torch.long
    )

    starts = [s for s, e in offsets]
    ends = [e for s, e in offsets]
    spans = []

    for s, e, lab in decode_spans(text, offsets, pred_ids.tolist()):
        ix = [
            i for i in range(bisect_right(ends, s), bisect_left(starts, e))
            if ends[i] > starts[i]
        ]
        conf = float(probs_t[ix, pred_ids[ix]].mean()) if ix else 0.0
        spans.append((int(s), int(e), lab, conf))

    return spans

# These are fixed comparisons, not a threshold search.
mixtures = {
    "champion": {"champion": 1.0},
    "B256": {"B256": 1.0},
    "targeted_pilot": {"targeted_pilot": 1.0},
    "champion_B256_50_50": {"champion": 0.5, "B256": 0.5},
    "champion_pilot_50_50": {
        "champion": 0.5, "targeted_pilot": 0.5
    },
    "all_three_equal": {
        "champion": 1/3, "B256": 1/3, "targeted_pilot": 1/3
    },
}

raw_predictions = {}
filtered_predictions = {}
metrics_by_method = {}

for method, weights in mixtures.items():
    raw = {}

    for doc_id, text in val_texts.items():
        combined = None
        for model_name, weight in weights.items():
            p = prob_cache[model_name][doc_id]
            combined = weight * p if combined is None else combined + weight * p

        # Average probabilities, not logits or extracted spans.
        combined = combined / sum(weights.values())

        # Renormalize defensively to guard against small numerical drift.
        combined = combined / np.clip(
            combined.sum(axis=1, keepdims=True), 1e-12, None
        )

        raw[doc_id] = probabilities_to_raw_spans(
            text, offset_cache[doc_id], combined
        )

    raw_predictions[method] = raw
    filtered = apply_thr(raw, FIXED_THRESHOLDS)
    filtered_predictions[method] = filtered

    metrics = calculate_exact_span_metrics(
        val_gold, filtered, labels=[
            "APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE",
            "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
        ]
    )
    metrics_by_method[method] = metrics

# ------------------------------------------------------------
# 4. Print validation results
# ------------------------------------------------------------
print("\n" + "=" * 78)
print("VALIDATION COMPARISON — FIXED CHAMPION THRESHOLDS")
print("=" * 78)
print(f"{'Method':<28} {'Macro-F1':>12}")
print("-" * 42)

for method, metrics in metrics_by_method.items():
    macro = metrics.get("MACRO_F1")
    if macro is None:
        macro = metrics.get("macro_f1")
    print(f"{method:<28} {float(macro):>12.6f}")

print("\nPer-label F1 for each method:")
for method, metrics in metrics_by_method.items():
    print(f"\n{method}")
    for label in [
        "APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE",
        "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
    ]:
        row = metrics.get(label, {})
        f1 = row.get("f1", row.get("F1"))
        if f1 is not None:
            print(f"  {label:<12} {float(f1):.4f}")

# ------------------------------------------------------------
# 5. Save a validation-only report; never overwrite predictions
# ------------------------------------------------------------
def json_safe_metrics(metrics):
    out = {}
    for key, value in metrics.items():
        if isinstance(value, dict):
            out[key] = {
                str(k): (
                    float(v) if isinstance(v, (np.floating, np.integer))
                    else v
                )
                for k, v in value.items()
            }
        elif isinstance(value, (np.floating, np.integer)):
            out[key] = float(value)
        else:
            out[key] = value
    return out

report = {
    "purpose": "validation-only probability ensemble comparison",
    "validation_document_count": len(val_texts),
    "thresholds": FIXED_THRESHOLDS,
    "methods": {
        name: json_safe_metrics(metrics)
        for name, metrics in metrics_by_method.items()
    },
    "note": (
        "No training or threshold tuning was performed. "
        "These are validation results, not test results."
    )
}

report_path = os.path.join(
    ARCHIVE, "probability_ensemble_validation_report.json"
)
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, default=str)

print("\nSaved validation report:", report_path)
print("No model checkpoint was overwritten.")
print("No test predictions were generated or modified.")


In [ ]:

# CELL 1 — Discover saved specialist checkpoints

from pathlib import Path
import os

SEARCH_ROOTS = [Path("/kaggle/working"), Path("/kaggle/input")]
ARCHIVE_DIR = Path("/kaggle/working/cs781_ensemble_archive")

terms = [
    "champion_ner.pt",
    "b256",
    "targeted_pilot",
    "name_swap",
    "nameswap",
    "mr_s0",
    "entity_replacement",
]

found = []
seen_paths = set()

for root in SEARCH_ROOTS:
    if not root.exists():
        continue
    for p in root.rglob("*.pt"):
        try:
            if p.stat().st_size > 700 * 1024**2:
                continue  # Skip Phase 1/Phase 4 LM-only checkpoints
        except OSError:
            continue

        low = p.name.lower()
        if any(term in low for term in terms):
            resolved = str(p.resolve())
            if resolved not in seen_paths:
                seen_paths.add(resolved)
                found.append(p)

print("SAVED NER CHECKPOINT CANDIDATES")
print("=" * 80)
for p in sorted(found, key=lambda x: (x.name.lower(), str(x))):
    print(f"{p.stat().st_size / 1024**2:7.1f} MiB | {p}")

name_swap = [
    p for p in found
    if any(x in p.name.lower() for x in ("name_swap", "nameswap", "mr_s0"))
]
if name_swap:
    print("\nName-swap checkpoint(s) found:")
    for p in name_swap:
        print(" ", p)
else:
    print(
        "\nNo saved name-swap checkpoint found. "
        "If its earlier run used save=False, its trained weights are not "
        "recoverable from the recorded metrics alone; this cell will not "
        "retrain it."
    )


In [ ]:

# CELL 2 — Specialist complementarity and span-union audit
# Validation only. No training, threshold search, or file overwrite.

import os
import json
import copy
import torch
from pathlib import Path
from collections import defaultdict

ARCHIVE_DIR = Path("/kaggle/working/cs781_ensemble_archive")
REPORT_PATH = ARCHIVE_DIR / "specialist_complementarity_audit.json"

required = [
    "champion_model", "predict_texts", "apply_thr",
    "calculate_exact_span_metrics", "val_texts", "val_gold",
    "id2label"
]
missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(f"Missing notebook variables/functions: {missing}")

if "VERIFIED_CHAMPION_THR" in globals():
    FIXED_THR = dict(VERIFIED_CHAMPION_THR)
elif "VERIFIED_THR" in globals():
    FIXED_THR = dict(VERIFIED_THR)
else:
    raise RuntimeError("Verified champion thresholds not found; stopping safely.")

def _safe_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        return torch.load(path, map_location="cpu")

def _state_from(payload):
    if isinstance(payload, dict):
        state = payload.get("model_state_dict", payload.get("state_dict"))
        if isinstance(state, dict):
            return state
    raise ValueError("Checkpoint has no recognizable model state dict.")

def _load_compatible_model(path):
    payload = _safe_load(path)
    state = _state_from(payload)

    labels = payload.get("bio_labels") if isinstance(payload, dict) else None
    expected_labels = [id2label[i] for i in sorted(id2label)]
    if labels is not None and list(labels) != expected_labels:
        raise ValueError(f"BIO label mismatch in {path}")

    candidate = copy.deepcopy(champion_model)
    candidate.load_state_dict(state, strict=True)
    candidate.to(next(champion_model.parameters()).device)
    candidate.eval()
    return candidate

# Only use the existing champion and checkpoint files.
model_paths = {
    "B256": ARCHIVE_DIR / "B256_candidate_20261011_170655.pt",
    "targeted_pilot": ARCHIVE_DIR / "targeted_pilot_8_steps.pt",
}

# Add any saved name-swap candidate discovered on disk.
for root in [Path("/kaggle/working"), Path("/kaggle/input")]:
    if not root.exists():
        continue
    for p in root.rglob("*.pt"):
        n = p.name.lower()
        if any(t in n for t in ("name_swap", "nameswap", "mr_s0")):
            try:
                if p.stat().st_size < 700 * 1024**2:
                    model_paths["name_swap"] = p
                    break
            except OSError:
                pass
    if "name_swap" in model_paths:
        break

# Normalize annotations to exact (document, start, end, label) keys.
def _gold_set_for_label(label):
    out = set()
    for doc_id, ents in val_gold.items():
        for ent in ents:
            if len(ent) >= 3 and str(ent[2]) == label:
                out.add((str(doc_id), int(ent[0]), int(ent[1]), label))
    return out

def _prediction_set(pred, label):
    out = set()
    for doc_id, spans in pred.items():
        for ent in spans:
            if len(ent) >= 3 and str(ent[2]) == label:
                out.add((str(doc_id), int(ent[0]), int(ent[1]), label))
    return out

# Score the champion with the exact same inference/threshold procedure.
champion_raw = predict_texts(champion_model, val_texts)
champion_pred = apply_thr(champion_raw, FIXED_THR)

candidate_predictions = {}
candidate_errors = {}

for name, path in model_paths.items():
    if not path.is_file():
        print(f"[SKIP] {name}: checkpoint not found at {path}")
        continue
    try:
        net = _load_compatible_model(path)
        raw = predict_texts(net, val_texts)
        candidate_predictions[name] = apply_thr(raw, FIXED_THR)
        net.to("cpu")
        del net
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"[OK] Evaluated {name}: {path}")
    except Exception as exc:
        candidate_errors[name] = repr(exc)
        print(f"[ERROR] {name}: {exc}")

LABELS_TO_AUDIT = [
    "RESP", "A.COUNSEL", "R.COUNSEL", "WIT", "JUDGE"
]

champion_sets = {
    lab: _prediction_set(champion_pred, lab)
    for lab in LABELS_TO_AUDIT
}
gold_sets = {
    lab: _gold_set_for_label(lab)
    for lab in LABELS_TO_AUDIT
}

audit = {
    "purpose": "validation-only model complementarity audit",
    "validation_documents": len(val_texts),
    "fixed_thresholds": FIXED_THR,
    "models": {},
    "candidate_errors": candidate_errors,
}

print("\n" + "=" * 100)
print("UNIQUE SPANS NOT ALREADY PREDICTED BY CHAMPION")
print("=" * 100)

for name, pred in candidate_predictions.items():
    model_report = {}
    print(f"\n### {name} ###")

    for lab in LABELS_TO_AUDIT:
        cand = _prediction_set(pred, lab)
        gold = gold_sets[lab]
        champ = champion_sets[lab]

        new_spans = cand - champ
        new_tp = new_spans & gold
        new_fp = new_spans - gold

        tp = len(cand & gold)
        fp = len(cand - gold)
        fn = len(gold - cand)
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

        row = {
            "candidate_predicted": len(cand),
            "candidate_TP": tp,
            "candidate_FP": fp,
            "candidate_FN": fn,
            "candidate_precision": precision,
            "candidate_recall": recall,
            "candidate_f1": f1,
            "new_spans_vs_champion": len(new_spans),
            "new_TP_vs_champion": len(new_tp),
            "new_FP_vs_champion": len(new_fp),
            "new_TP_examples": [
                list(x) for x in sorted(new_tp)[:10]
            ],
            "new_FP_examples": [
                list(x) for x in sorted(new_fp)[:10]
            ],
        }
        model_report[lab] = row

        print(
            f"{lab:<12} candidate F1={f1:.4f} P={precision:.3f} "
            f"new_TP={len(new_tp):3d} new_FP={len(new_fp):3d} "
            f"new_spans={len(new_spans):3d}"
        )

    audit["models"][name] = model_report

# ------------------------------------------------------------
# Compare candidate-only unions without changing the champion.
# First: union every label from one candidate with the champion.
# Second: union target labels only from all candidates.
# ------------------------------------------------------------
def _union_predictions(base, extras, labels_only=None):
    out = {str(d): set() for d in base}
    for d, spans in base.items():
        out.setdefault(str(d), set()).update(
            (int(s), int(e), str(lab)) for s, e, lab in spans
        )

    for pred in extras:
        for d, spans in pred.items():
            out.setdefault(str(d), set())
            for s, e, lab in spans:
                lab = str(lab)
                if labels_only is None or lab in labels_only:
                    out[str(d)].add((int(s), int(e), lab))

    return {d: sorted(spans) for d, spans in out.items()}

def _macro_f1(pred):
    m = calculate_exact_span_metrics(
        val_gold, pred,
        labels=[
            "APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE",
            "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
        ]
    )
    return float(m["MACRO_F1"]), m

champ_macro, champ_metrics = _macro_f1(champion_pred)
print("\n" + "=" * 100)
print("VALIDATION MACRO-F1: CHAMPION VS SPAN UNIONS")
print("=" * 100)
print(f"{'Champion baseline':<36} {champ_macro:.6f}")

audit["union_comparisons"] = {
    "champion": {"macro_f1": champ_macro}
}

for name, pred in candidate_predictions.items():
    union = _union_predictions(champion_pred, [pred])
    score, metrics = _macro_f1(union)
    audit["union_comparisons"][f"champion_plus_{name}_all_labels"] = {
        "macro_f1": score,
        "delta_vs_champion": score - champ_macro,
    }
    print(
        f"{('Champion + ' + name + ' (all labels)'):<36} "
        f"{score:.6f} ({score-champ_macro:+.6f})"
    )

target_labels = set(LABELS_TO_AUDIT)
target_union = _union_predictions(
    champion_pred,
    list(candidate_predictions.values()),
    labels_only=target_labels
)
target_score, target_metrics = _macro_f1(target_union)
audit["union_comparisons"]["champion_plus_all_candidates_target_labels"] = {
    "macro_f1": target_score,
    "delta_vs_champion": target_score - champ_macro,
}
print(
    f"{'Champion + all candidates (target labels)':<36} "
    f"{target_score:.6f} ({target_score-champ_macro:+.6f})"
)

# Save audit only. This does not write any prediction JSON.
with open(REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(audit, f, indent=2, default=str)

print("\nAudit saved:", REPORT_PATH)
print("No training performed.")
print("Champion checkpoint and prediction files were not modified.")
print("All scores above are validation results, not test results.")


In [ ]:

# CELL 3 — Expanded checkpoint audit
# Evaluates every unique, compatible saved NER checkpoint found.
# Validation only; does not train or overwrite predictions.

from pathlib import Path
from collections import defaultdict
import hashlib, json, copy, os, gc
import torch

ROOTS = [Path("/kaggle/working"), Path("/kaggle/input")]
ARCHIVE = Path("/kaggle/working/cs781_ensemble_archive")
REPORT = ARCHIVE / "expanded_checkpoint_audit.json"
ARCHIVE.mkdir(parents=True, exist_ok=True)

def _load_payload(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        return torch.load(path, map_location="cpu")

def _extract_state(payload):
    if not isinstance(payload, dict):
        return None
    for key in ("model_state_dict", "state_dict"):
        if isinstance(payload.get(key), dict):
            return payload[key]
    return None

def _state_hash(state):
    h = hashlib.sha256()
    for k in sorted(state):
        v = state[k]
        if torch.is_tensor(v):
            v = v.detach().cpu().contiguous()
            h.update(k.encode())
            h.update(str(tuple(v.shape)).encode())
            h.update(v.numpy().tobytes())
    return h.hexdigest()

def _is_ner_state(state):
    if not state:
        return False
    cw = state.get("classifier.weight")
    return (
        torch.is_tensor(cw)
        and tuple(cw.shape) == (25, 768)
        and "encoder.wte.weight" in state
    )

# Gather all small checkpoint files, not just names we already know.
paths = []
for root in ROOTS:
    if not root.exists():
        continue
    for p in root.rglob("*"):
        if not p.is_file() or p.suffix.lower() not in {".pt", ".pth", ".ckpt"}:
            continue
        try:
            size = p.stat().st_size
        except OSError:
            continue
        # Skip large Phase-1/Phase-4 language-model checkpoints.
        if size <= 700 * 1024**2:
            paths.append(p)

# Deduplicate filesystem aliases and exact model states.
unique_paths = {}
for p in paths:
    try:
        unique_paths[str(p.resolve())] = p
    except OSError:
        unique_paths[str(p)] = p

seen_hashes = {}
candidate_paths = []
inventory = []

for p in sorted(unique_paths.values(), key=lambda x: str(x)):
    try:
        payload = _load_payload(p)
        state = _extract_state(payload)
        if not _is_ner_state(state):
            continue

        # Don't classify random/initialization states as trained models.
        low_name = p.name.lower()
        init_like = any(
            term in low_name
            for term in ("model_init", "initial_state", "initial_checkpoint")
        )

        fp = _state_hash(state)
        item = {
            "path": str(p),
            "name": p.name,
            "fingerprint": fp,
            "initialization_like_name": init_like,
            "duplicate_of": seen_hashes.get(fp),
        }
        inventory.append(item)

        if fp not in seen_hashes and not init_like:
            seen_hashes[fp] = str(p)
            candidate_paths.append((p, fp))
        del payload, state
    except Exception as exc:
        print(f"[SKIP] {p}: {exc}")

print("Compatible checkpoint inventory:")
for item in inventory:
    print(
        f"- {item['name']} | "
        f"{'initialization-like' if item['initialization_like_name'] else 'candidate/duplicate'}"
        + (f" | duplicate of {item['duplicate_of']}" if item["duplicate_of"] else "")
    )

# Score each unique saved trained model using the same fixed thresholds.
candidate_preds = {}
candidate_details = {}
device = next(champion_model.parameters()).device

for p, fp in candidate_paths:
    # Don't evaluate champion twice; it is the baseline already.
    try:
        payload = _load_payload(p)
        state = _extract_state(payload)

        if not _is_ner_state(state):
            continue

        if fp == _state_hash(champion_model.state_dict()):
            print(f"[BASELINE] {p.name}: identical to in-memory champion")
            continue

        labels = payload.get("bio_labels")
        expected = [id2label[i] for i in sorted(id2label)]
        if labels is not None and list(labels) != expected:
            print(f"[SKIP LABEL MISMATCH] {p}")
            continue

        net = copy.deepcopy(champion_model)
        net.load_state_dict(state, strict=True)
        net.to(device).eval()

        raw = predict_texts(net, val_texts)
        pred = apply_thr(raw, FIXED_THR)
        name = f"{p.stem}::{fp[:8]}"
        candidate_preds[name] = pred
        candidate_details[name] = {
            "path": str(p),
            "fingerprint": fp,
        }

        print(f"[EVALUATED] {name}")

        del net, payload, state, raw
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as exc:
        print(f"[EVALUATION ERROR] {p}: {repr(exc)}")

# Build exact-span sets for candidate complementarity.
AUDIT_LABELS = ["RESP", "A.COUNSEL", "R.COUNSEL", "WIT", "JUDGE"]

def _pred_set(pred, label):
    return {
        (str(d), int(s), int(e), str(lab))
        for d, spans in pred.items()
        for s, e, lab in spans
        if str(lab) == label
    }

def _gold_set(label):
    return {
        (str(d), int(ent[0]), int(ent[1]), str(ent[2]))
        for d, spans in val_gold.items()
        for ent in spans
        if len(ent) >= 3 and str(ent[2]) == label
    }

champion_sets = {
    lab: _pred_set(champion_pred, lab) for lab in AUDIT_LABELS
}
gold_sets = {lab: _gold_set(lab) for lab in AUDIT_LABELS}

audit_results = {
    "inventory": inventory,
    "candidate_models": candidate_details,
    "models": {},
    "note": (
        "Only compatible saved states were evaluated. Metrics are validation-only. "
        "Missing unsaved experiments cannot be reconstructed from metric rows."
    ),
}

print("\nUNIQUE CONTRIBUTIONS VS CHAMPION")
print("=" * 105)

for name, pred in candidate_preds.items():
    rows = {}
    print("\n", name)
    for lab in AUDIT_LABELS:
        cand = _pred_set(pred, lab)
        gold = gold_sets[lab]
        new = cand - champion_sets[lab]
        new_tp = new & gold
        new_fp = new - gold
        tp = len(cand & gold)
        fp = len(cand - gold)
        fn = len(gold - cand)
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )
        rows[lab] = {
            "candidate_pred": len(cand),
            "candidate_TP": tp,
            "candidate_FP": fp,
            "candidate_FN": fn,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "new_TP_vs_champion": len(new_tp),
            "new_FP_vs_champion": len(new_fp),
            "new_TP_examples": [list(x) for x in sorted(new_tp)[:8]],
            "new_FP_examples": [list(x) for x in sorted(new_fp)[:8]],
        }
        print(
            f"{lab:<12} F1={f1:.4f} "
            f"new_TP={len(new_tp):3d} new_FP={len(new_fp):3d}"
        )
    audit_results["models"][name] = rows

# Candidate-only unions restricted to the target labels.
def _union_target(base, extras, labels):
    out = {
        str(d): {
            (int(s), int(e), str(lab)) for s, e, lab in spans
        }
        for d, spans in base.items()
    }
    for pred in extras:
        for d, spans in pred.items():
            out.setdefault(str(d), set())
            out[str(d)].update(
                (int(s), int(e), str(lab))
                for s, e, lab in spans
                if str(lab) in labels
            )
    return {d: sorted(v) for d, v in out.items()}

def _macro(pred):
    m = calculate_exact_span_metrics(val_gold, pred, ENTITY_LABELS)
    return float(m["MACRO_F1"])

champ_macro = _macro(champion_pred)
union = _union_target(
    champion_pred,
    list(candidate_preds.values()),
    set(AUDIT_LABELS)
)
union_macro = _macro(union)

audit_results["summary"] = {
    "champion_macro_f1_fixed_thresholds": champ_macro,
    "champion_plus_all_unique_checkpoint_candidates_target_labels_macro_f1": union_macro,
    "union_delta_vs_champion": union_macro - champ_macro,
}

with open(REPORT, "w", encoding="utf-8") as f:
    json.dump(audit_results, f, indent=2, default=str)

print("\n" + "=" * 105)
print(f"Champion fixed-threshold macro-F1: {champ_macro:.6f}")
print(f"Champion + all candidate target spans: {union_macro:.6f}")
print(f"Delta: {union_macro - champ_macro:+.6f}")
print("Report saved:", REPORT)
print("No training performed; no predictions/checkpoints overwritten.")


In [ ]:

# ============================================================
# CELL 1 — Recover missing experiment checkpoints
# Exact recorded recipes; no new hyperparameter search.
#
# B512_s0       seed=0, L=512, stride=448, no augmentation
# B512_s1       seed=1, L=512, stride=448, no augmentation
# ECHO256_s0    seed=0, L=256, stride=192, echo=True
# B512_MR_s0    seed=0, L=512, stride=448, name-swap K=2, p=0.5
#
# All checkpoints use unique filenames. Champion is not modified.
# ============================================================

import os, json, gc, torch
from pathlib import Path
from datetime import datetime

assert "run_cfg" in globals(), (
    "run_cfg is missing. Run the experiment-harness definition cell first."
)
assert "BIO_LABELS" in globals(), "BIO_LABELS is missing."
assert "champion_model" in globals(), "Champion model is missing."

EXP_DIR = Path("/kaggle/working/exps_v2")
EXP_DIR.mkdir(parents=True, exist_ok=True)
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

RECOVERY_JOBS = [
    {
        "name": f"RECOVER_B512_s0_{STAMP}",
        "original_experiment": "B512_s0",
        "cfg": {"seed": 0, "L": 512, "stride": 448,
                "echo": False, "aug": None, "save": True},
    },
    {
        "name": f"RECOVER_B512_s1_{STAMP}",
        "original_experiment": "B512_s1",
        "cfg": {"seed": 1, "L": 512, "stride": 448,
                "echo": False, "aug": None, "save": True},
    },
    {
        "name": f"RECOVER_ECHO256_s0_{STAMP}",
        "original_experiment": "ECHO256_s0",
        "cfg": {"seed": 0, "L": 256, "stride": 192,
                "echo": True, "aug": None, "save": True},
    },
    {
        "name": f"RECOVER_B512_MR_s0_{STAMP}",
        "original_experiment": "B512_MR_s0",
        "cfg": {"seed": 0, "L": 512, "stride": 448,
                "echo": False, "aug": {"K": 2, "p": 0.5},
                "save": True},
    },
]

recovery_manifest = {
    "created_at": STAMP,
    "purpose": "checkpoint recovery by rerunning original recipes",
    "test_evaluated": False,
    "models": [],
}

for job in RECOVERY_JOBS:
    name = job["name"]
    path = EXP_DIR / f"{name}.pt"

    if path.exists():
        raise FileExistsError(
            f"Refusing to overwrite existing checkpoint: {path}"
        )

    print("\n" + "=" * 78)
    print("RECOVERING:", job["original_experiment"])
    print("Checkpoint:", path)
    print("Config:", job["cfg"])
    print("=" * 78)

    # Runs the recorded experiment recipe; save=True preserves its EMA weights.
    result = run_cfg(name, **job["cfg"])

    if not path.exists():
        raise RuntimeError(
            f"run_cfg completed but expected checkpoint is missing: {path}"
        )

    # Add BIO-label metadata to the newly created checkpoint.
    try:
        payload = torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        payload = torch.load(path, map_location="cpu")

    payload["bio_labels"] = list(BIO_LABELS)
    payload["entity_labels"] = list(ENTITY_LABELS)
    payload["recovery_metadata"] = {
        "original_experiment": job["original_experiment"],
        "recovery_run_name": name,
        "config": job["cfg"],
        "created_at": STAMP,
        "test_evaluated": False,
    }

    # Write to a temporary file first, then replace only this new checkpoint.
    temp_path = path.with_suffix(".tmp")
    torch.save(payload, temp_path)
    os.replace(temp_path, path)

    row = {
        "original_experiment": job["original_experiment"],
        "recovery_run_name": name,
        "checkpoint": str(path),
        "size_bytes": path.stat().st_size,
        "cfg": job["cfg"],
        "recorded_run_result": {
            k: v for k, v in result.items()
            if k not in ("predictions", "model")
        },
    }
    recovery_manifest["models"].append(row)

    print("[SAVED]", path)
    print(
        "Recorded harness hybrid with champion thresholds:",
        result.get("hybrid_champ_thr")
    )

    del payload, result
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

manifest_path = ARCHIVE_DIR = Path(
    "/kaggle/working/cs781_ensemble_archive"
)
manifest_path.mkdir(parents=True, exist_ok=True)
manifest_path = manifest_path / f"recovered_harness_models_{STAMP}.json"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(recovery_manifest, f, indent=2, default=str)

print("\nRECOVERY COMPLETE")
print("Manifest:", manifest_path)
for item in recovery_manifest["models"]:
    print("-", item["original_experiment"], "->", item["checkpoint"])

print("\nChampion weights and existing prediction JSON files were not modified.")


In [ ]:

# ============================================================
# CELL 2 — Recover weighted-CE and focal-loss checkpoints
# Exact recorded experiment recipe; one epoch per variant.
# Saves both models, irrespective of validation outcome.
# ============================================================

import os, json, gc, copy, random
from pathlib import Path
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

required = [
    "champion_model", "train_texts", "train_gold",
    "val_texts", "val_gold", "tokenizer",
    "BIO_LABELS", "label2id", "ENTITY_LABELS",
    "exp_full_pipeline", "device",
]
missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(f"Missing notebook variables: {missing}")

STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = Path("/kaggle/working/cs781_ensemble_archive")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Exact settings from the original weighted-CE/focal experiment.
SEED = 78115
MAX_LEN = 512
STRIDE = 448
BATCH_SIZE = 1
EPOCHS = 1
ENCODER_LR = 5e-6
CLASSIFIER_LR = 2e-5
FOCAL_GAMMA = 2.0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Confirm that the champion reference is still the same hybrid baseline.
champion_model.eval()
_, reference_metrics = exp_full_pipeline(
    champion_model, val_texts, val_gold
)
reference_score = float(reference_metrics["MACRO_F1"])

print("Champion hybrid validation macro-F1:", round(reference_score, 6))
if abs(reference_score - 0.333307) > 0.002:
    raise RuntimeError(
        "Reference hybrid score does not reproduce approximately 0.333307. "
        "Stop and investigate before training."
    )

# ------------------------------------------------------------
# Build the same BIO chunks used by the original loss experiment.
# Each token is supervised once; overlaps are masked.
# ------------------------------------------------------------
def _encode_loss_experiment_doc(text, entities):
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids, offsets = list(enc.ids), list(enc.offsets)

    tags = [label2id["O"]] * len(ids)
    conflicts = [False] * len(ids)

    for ent in entities:
        if len(ent) < 3:
            continue

        s, e, lab = int(ent[0]), int(ent[1]), str(ent[2])
        if lab not in ENTITY_LABELS or s < 0 or e <= s or e > len(text):
            continue

        token_indices = [
            i for i, (a, b) in enumerate(offsets)
            if b > a and a < e and b > s
        ]

        for j, i in enumerate(token_indices):
            tag_id = label2id[("B-" if j == 0 else "I-") + lab]
            if tags[i] != label2id["O"] and tags[i] != tag_id:
                conflicts[i] = True
            else:
                tags[i] = tag_id

    for i, conflict in enumerate(conflicts):
        if conflict:
            tags[i] = -100

    chunks, seen = [], set()
    for start in range(0, len(ids), STRIDE):
        end = min(start + MAX_LEN, len(ids))
        if start >= end:
            break

        chunk_labels = []
        for i in range(start, end):
            if i in seen:
                chunk_labels.append(-100)
            else:
                chunk_labels.append(tags[i])
                seen.add(i)

        chunks.append({
            "input_ids": ids[start:end],
            "labels": chunk_labels,
        })

        if end == len(ids):
            break

    return chunks, tags

train_chunks = []
tag_counts = torch.zeros(len(BIO_LABELS), dtype=torch.long)

for doc_id, text in train_texts.items():
    chunks, full_tags = _encode_loss_experiment_doc(
        text, train_gold[str(doc_id)]
    )
    train_chunks.extend(chunks)
    for tag in full_tags:
        if tag >= 0:
            tag_counts[tag] += 1

assert train_chunks, "No training chunks were generated."

def _collate_loss_batch(batch):
    n = max(len(x["input_ids"]) for x in batch)
    ids = torch.zeros((len(batch), n), dtype=torch.long)
    mask = torch.zeros_like(ids)
    labels = torch.full((len(batch), n), -100, dtype=torch.long)

    for i, item in enumerate(batch):
        k = len(item["input_ids"])
        ids[i, :k] = torch.tensor(item["input_ids"], dtype=torch.long)
        mask[i, :k] = 1
        labels[i, :k] = torch.tensor(item["labels"], dtype=torch.long)

    return ids, mask, labels

generator = torch.Generator().manual_seed(SEED)
train_loader_loss = DataLoader(
    train_chunks,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=_collate_loss_batch,
    generator=generator,
)

# Same bounded inverse-frequency alpha weights.
class_weights = tag_counts.float().clamp_min(1).pow(-0.25)
class_weights = (
    class_weights / class_weights.mean()
).clamp(0.5, 3.0).to(device)

def _focal_token_loss(logits, labels, alpha, gamma=2.0):
    flat_logits = logits.reshape(-1, logits.shape[-1])
    flat_labels = labels.reshape(-1)
    valid = flat_labels != -100

    if not valid.any():
        return flat_logits.sum() * 0.0

    logits_valid = flat_logits[valid]
    labels_valid = flat_labels[valid]

    log_probs = torch.log_softmax(logits_valid, dim=-1)
    log_pt = log_probs.gather(1, labels_valid[:, None]).squeeze(1)
    pt = log_pt.exp()
    a = alpha[labels_valid]

    return (
        -a * ((1.0 - pt).clamp_min(0) ** gamma) * log_pt
    ).mean()

def _run_and_save_loss_model(name, loss_kind):
    # Start each variant from an independent copy of the untouched champion.
    net = copy.deepcopy(champion_model).to(device)
    net.train()

    optimizer = torch.optim.AdamW(
        [
            {"params": net.encoder.parameters(), "lr": ENCODER_LR},
            {"params": net.classifier.parameters(), "lr": CLASSIFIER_LR},
        ],
        weight_decay=0.01,
    )

    ce_loss = nn.CrossEntropyLoss(
        weight=class_weights, ignore_index=-100
    )
    losses = []

    for epoch in range(EPOCHS):
        net.train()
        for ids, mask, labels in tqdm(
            train_loader_loss,
            desc=f"{name} epoch {epoch + 1}/{EPOCHS}",
        ):
            ids = ids.to(device)
            mask = mask.to(device)
            labels = labels.to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = net(ids, mask)

            if loss_kind == "weighted_ce":
                loss = ce_loss(
                    logits.reshape(-1, len(BIO_LABELS)),
                    labels.reshape(-1),
                )
            elif loss_kind == "focal":
                loss = _focal_token_loss(
                    logits, labels, class_weights, gamma=FOCAL_GAMMA
                )
            else:
                raise ValueError(loss_kind)

            if not torch.isfinite(loss):
                raise RuntimeError(f"{name}: non-finite training loss.")

            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            optimizer.step()
            losses.append(float(loss.item()))

    net.eval()
    val_predictions, val_metrics = exp_full_pipeline(
        net, val_texts, val_gold
    )
    score = float(val_metrics["MACRO_F1"])

    # Save regardless of score, because this is an experiment archive.
    path = OUT_DIR / f"{name}_{STAMP}.pt"
    if path.exists():
        raise FileExistsError(f"Refusing to overwrite: {path}")

    torch.save({
        "model_state_dict": {
            k: v.detach().cpu().clone()
            for k, v in net.state_dict().items()
        },
        "bio_labels": list(BIO_LABELS),
        "entity_labels": list(ENTITY_LABELS),
        "experiment": name,
        "loss_kind": loss_kind,
        "seed": SEED,
        "epochs": EPOCHS,
        "max_len": MAX_LEN,
        "stride": STRIDE,
        "batch_size": BATCH_SIZE,
        "encoder_lr": ENCODER_LR,
        "classifier_lr": CLASSIFIER_LR,
        "focal_gamma": FOCAL_GAMMA if loss_kind == "focal" else None,
        "class_weight_formula": "inverse frequency ^ -0.25, normalized, clipped [0.5, 3.0]",
        "training_loss_history": losses,
        "validation_macro_f1": score,
        "reference_macro_f1": reference_score,
        "validation_delta": score - reference_score,
        "validation_metrics": val_metrics,
        "test_evaluated": False,
    }, path)

    print("\nSaved:", path)
    print(f"Validation macro-F1: {score:.6f}")
    print(f"Delta vs reference:  {score-reference_score:+.6f}")
    print("Mean training loss:", float(np.mean(losses)))

    # Per-entity results retained in the checkpoint and shown here.
    for label in ENTITY_LABELS:
        m = val_metrics[label]
        print(
            f"{label:12s} P={m['precision']:.4f} "
            f"R={m['recall']:.4f} F1={m['f1']:.4f} "
            f"TP={m['TP']} FP={m['FP']} FN={m['FN']}"
        )

    del optimizer, net
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "name": name,
        "checkpoint": str(path),
        "validation_macro_f1": score,
        "delta_vs_reference": score - reference_score,
        "mean_training_loss": float(np.mean(losses)),
    }

loss_recovery_results = []

loss_recovery_results.append(
    _run_and_save_loss_model(
        "recovered_weighted_cross_entropy", "weighted_ce"
    )
)
loss_recovery_results.append(
    _run_and_save_loss_model(
        "recovered_focal_gamma2", "focal"
    )
)

summary_path = OUT_DIR / f"loss_experiment_recovery_{STAMP}.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump({
        "reference_macro_f1": reference_score,
        "results": loss_recovery_results,
        "test_evaluated": False,
    }, f, indent=2)

print("\nLOSS EXPERIMENT RECOVERY COMPLETE")
print("Summary:", summary_path)
for row in loss_recovery_results:
    print(
        f"- {row['name']}: {row['checkpoint']} "
        f"(macro-F1={row['validation_macro_f1']:.6f})"
    )

print("\nChampion checkpoint and existing prediction JSON files were not modified.")
print("No test labels were used.")


In [ ]:

# CELL 3 — Final recovered-model inventory

from pathlib import Path
import torch

roots = [
    Path("/kaggle/working/exps_v2"),
    Path("/kaggle/working/cs781_ensemble_archive"),
]

print("RECOVERED AND ARCHIVED NER CHECKPOINTS")
print("=" * 100)

for root in roots:
    if not root.exists():
        continue
    for path in sorted(root.glob("*.pt")):
        try:
            if path.stat().st_size > 700 * 1024**2:
                continue
            try:
                payload = torch.load(
                    path, map_location="cpu", weights_only=True
                )
            except TypeError:
                payload = torch.load(path, map_location="cpu")

            state = payload.get("model_state_dict", {})
            cw = state.get("classifier.weight") if isinstance(state, dict) else None

            if torch.is_tensor(cw) and tuple(cw.shape) == (25, 768):
                cfg = payload.get("cfg", {})
                metadata = payload.get("recovery_metadata", {})
                print(f"\n{path}")
                print("  size MiB:", round(path.stat().st_size / 1024**2, 1))
                print("  fingerprint source keys:", len(state))
                print("  experiment:", metadata.get("original_experiment",
                                                    payload.get("experiment", cfg.get("name", "unknown"))))
                print("  config:", cfg or metadata.get("config", "see checkpoint metadata"))
                print("  validation macro-F1:",
                      payload.get("validation_macro_f1", "see run ledger/summary"))
        except Exception as exc:
            print("[SKIP]", path, repr(exc))
